# TCGA Bioinformatics Paper 2: Multilevel Feature Stability Analysis

**Purpose.** This notebook extends the dissertation analyses to quantify several distinct forms of feature stability rather than relying on exact gene overlap alone. It uses the fixed TCGA-LGG molecular-subtype analysis and the fixed TCGA kidney TPM/CPM analysis as completed inputs, then performs reproducible secondary analyses without reacquiring TCGA data.

**Revision 3 additions.** This version retains all Revision 2 analyses and adds **restartable NSGA-II checkpoints for the long patient-sampling experiment**. During each incomplete publication replicate, the full live `pymoo` algorithm state is serialized every 25 completed generations using compressed `dill` checkpoints. A previous checkpoint is retained as a backup, checkpoint writes are atomic, and a restarted notebook automatically resumes from the newest valid compatible checkpoint. Completed replicates are still detected and skipped. Progress written after the last checkpoint is treated as diagnostic only and is safely archived before resume.

The notebook is intended for a **Google Colab front end connected to a Windows local runtime** and retains controlled multicore processing for computationally expensive null, bootstrap, and NSGA-II calculations.


**Revision 5 additions.** After completion of all 20 publication patient-resampling runs, this version adds a self-contained final post-analysis cell that does **not** rerun NSGA-II. It evaluates every resampled gene panel on the untouched 85-patient validation cohort under two training designs, quantifies stability using the first 5, 10, 15, and 20 resamples, and creates formal 20-resample gene-recurrence tables and manuscript-ready figures.


## 1. Package check

This cell only installs packages that are missing from the local Python environment. The notebook is designed for a **Google Colab front end connected to the Windows local runtime**, as in the dissertation analyses.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "joblib": "joblib",
    "threadpoolctl": "threadpoolctl",
    "psutil": "psutil",
    "networkx": "networkx",
    "pymoo": "pymoo",
    "dill": "dill",
}

missing = [pip_name for module, pip_name in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

## 2. Imports, paths, analysis settings, and output folders

All requested inputs and all new outputs remain beneath:

`C:\Users\we3mo\Dropbox\MY DATA DRIVE\CANCER\2026 - ADD ANALYSIS FOR TCGA BIOINFORMATICS_2`

The notebook automatically tolerates one extra nested directory created by ZIP extraction, so the input folders do not need further reorganization.

In [ ]:
from pathlib import Path
from IPython.display import display
from itertools import combinations
from collections import defaultdict

import json
import gzip
import hashlib
import shutil
import math
import os
import platform
import random
import tempfile
import time
import warnings

import dill
import joblib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import psutil

from joblib import Parallel, delayed
from scipy.optimize import linear_sum_assignment
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching
from scipy.stats import pearsonr
from threadpoolctl import threadpool_limits

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

NOTEBOOK_REVISION = "2026-09-26-PAPER2-FEATURE-STABILITY-LOCAL-MULTICORE-v5-FINAL-POST-ANALYSIS"

DEFAULT_BASE_DIR = Path(
    r"C:\Users\we3mo\Dropbox\MY DATA DRIVE\CANCER"
    r"\2026 - ADD ANALYSIS FOR TCGA BIOINFORMATICS_2"
)
# PAPER2_BASE_DIR is only needed for portability/testing. On the user's Windows
# local runtime the default path above is used automatically.
BASE_DIR = Path(os.environ.get("PAPER2_BASE_DIR", str(DEFAULT_BASE_DIR)))

LGG_DIR_REQUESTED = BASE_DIR / "TCGA_LGG_MOLECULAR_SUBTYPE_FINAL_ANALYSIS"
KIDNEY_PROCESSED_REQUESTED = BASE_DIR / "TCGA_Kidney_processed_outputs"
KIDNEY_MODELING_REQUESTED = BASE_DIR / "TCGA_Kidney_modeling_outputs"

print("Operating system:", platform.system())
print("Base directory:", BASE_DIR)
if not BASE_DIR.exists():
    raise FileNotFoundError(
        "The Paper 2 base directory is not visible to this Python runtime:\n"
        f"{BASE_DIR}\n\n"
        "Confirm that Google Colab is connected to the Windows local runtime and that "
        "the base folder exists before running the notebook."
    )

OUTPUT_DIR = BASE_DIR / "PAPER2_OUTPUTS"
INPUT_VALIDATION_DIR = OUTPUT_DIR / "01_INPUT_VALIDATION"
EXACT_DIR = OUTPUT_DIR / "02_EXACT_GENE_STABILITY"
NOGUEIRA_DIR = OUTPUT_DIR / "03_NOGUEIRA_STABILITY"
CORR_DIR = OUTPUT_DIR / "04_CORRELATION_ADJUSTED_STABILITY"
PRESERVATION_DIR = OUTPUT_DIR / "05_TRAIN_VALIDATION_PRESERVATION"
NULL_DIR = OUTPUT_DIR / "06_NULL_MODEL"
BOOTSTRAP_DIR = OUTPUT_DIR / "07_BOOTSTRAP_STABILITY"
KIDNEY_DIR = OUTPUT_DIR / "08_KIDNEY_TPM_CPM_STABILITY"
RESAMPLE_DIR = OUTPUT_DIR / "09_SAMPLE_PERTURBATION"
PROGRAM_DIR = OUTPUT_DIR / "10_CORRELATION_COMPONENT_STABILITY"
PREDICTIVE_DIR = OUTPUT_DIR / "11_PREDICTIVE_STABILITY"
FIGURE_DIR = OUTPUT_DIR / "FIGURES"
TABLE_DIR = OUTPUT_DIR / "TABLES"
CACHE_DIR = OUTPUT_DIR / "CACHE"
JOBLIB_TEMP_DIR = Path(tempfile.gettempdir()) / "TCGA_Paper2_Stability_joblib"

for directory in [
    OUTPUT_DIR, INPUT_VALIDATION_DIR, EXACT_DIR, NOGUEIRA_DIR, CORR_DIR,
    PRESERVATION_DIR, NULL_DIR, BOOTSTRAP_DIR, KIDNEY_DIR, RESAMPLE_DIR,
    PROGRAM_DIR, PREDICTIVE_DIR, FIGURE_DIR, TABLE_DIR, CACHE_DIR, JOBLIB_TEMP_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Parallel settings
# -----------------------------
N_WORKERS = 10
THREADS_PER_WORKER = 1
JOBLIB_BACKEND = "loky"

logical = psutil.cpu_count(logical=True) or os.cpu_count() or 1
if logical < N_WORKERS:
    print(f"WARNING: only {logical} logical processors detected; reducing N_WORKERS to {logical}.")
    N_WORKERS = max(1, int(logical))

# -----------------------------
# Primary stability settings
# -----------------------------
LGG_SEEDS = [42, 101, 202, 303, 404]
CORRELATION_THRESHOLDS = [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
PRIMARY_CORRELATION_THRESHOLD = 0.80
NETWORK_MIN_COMPONENT_SIZE = 4
RANDOM_SEED = 42

# Publication values. PAPER2_TEST_MODE=1 automatically uses small values for
# code validation without changing the saved notebook defaults.
TEST_MODE = os.environ.get("PAPER2_TEST_MODE", "0") == "1"
N_NULL_REPLICATES = 50 if TEST_MODE else 10_000
N_CORRELATION_BOOTSTRAPS = 50 if TEST_MODE else 2_000

DISSERTATION_NAVY = "#001544"
DISSERTATION_TEAL = "#006D77"
FIGURE_DPI = 300

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Notebook revision:", NOTEBOOK_REVISION)
print("Base directory:", BASE_DIR)
print("Output directory:", OUTPUT_DIR)
print("Workers:", N_WORKERS)
print("Null replicates:", N_NULL_REPLICATES)
print("Correlation bootstraps:", N_CORRELATION_BOOTSTRAPS)
print("Test mode:", TEST_MODE)


## 3. Helper functions for input discovery and reproducible output

In [ ]:
def resolve_root(requested_root, required_filename):
    """Accept a renamed folder or one extra ZIP-created nested folder."""
    requested_root = Path(requested_root)
    if (requested_root / required_filename).exists():
        return requested_root
    if not requested_root.exists():
        raise FileNotFoundError(f"Input directory not found: {requested_root}")
    candidates = [p for p in requested_root.iterdir()
                  if p.is_dir() and (p / required_filename).exists()]
    if len(candidates) == 1:
        print(f"Using nested input directory: {candidates[0]}")
        return candidates[0]
    raise FileNotFoundError(
        f"Could not locate {required_filename} directly under {requested_root} "
        "or under exactly one immediate child directory."
    )


def save_figure(fig, stem, directory=FIGURE_DIR):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    fig.savefig(directory / f"{stem}.png", dpi=FIGURE_DPI, bbox_inches="tight")
    fig.savefig(directory / f"{stem}.pdf", bbox_inches="tight")


def write_json(obj, path):
    def convert(value):
        if isinstance(value, np.generic):
            return value.item()
        if isinstance(value, Path):
            return str(value)
        if isinstance(value, dict):
            return {str(k): convert(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return [convert(v) for v in value]
        return value
    Path(path).write_text(json.dumps(convert(obj), indent=2), encoding="utf-8")


def split_semicolon(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [x for x in str(value).split(";") if x]


def gene_symbol(gene_id, mapping):
    symbol = str(mapping.get(gene_id, "")).strip()
    return symbol if symbol else str(gene_id)


def require_files(root, relative_paths, dataset_name):
    records = []
    missing = []
    for rel in relative_paths:
        path = Path(root) / rel
        exists = path.exists()
        records.append({"dataset": dataset_name, "relative_path": rel, "exists": exists,
                        "size_bytes": path.stat().st_size if exists else np.nan})
        if not exists:
            missing.append(str(path))
    if missing:
        raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))
    return pd.DataFrame(records)

## 4. Validate and load the fixed TCGA-LGG inputs

No TCGA data are reacquired and no filtering or MAD ranking is repeated. The notebook starts from the exact 197-patient training matrix, 85-patient held-out matrix, fixed 3,000-gene candidate universe, and five saved selected NSGA-II panels from Chapter 12.

In [ ]:
LGG_DIR = resolve_root(LGG_DIR_REQUESTED, "patient_split_assignments.csv")

lgg_required = [
    "patient_split_assignments.csv",
    "NSGA_candidate_gene_pool.csv",
    "training_candidate_expression_log2TPM1.csv.gz",
    "heldout_validation_candidate_expression_log2TPM1.csv.gz",
    "analysis_configuration.json",
    "tables/selected_pareto_solution_by_seed.csv",
]
for seed in LGG_SEEDS:
    lgg_required.extend([
        f"nsga_multiseed_runs/seed_{seed}/selected_genes_seed_{seed}.csv",
        f"nsga_multiseed_runs/seed_{seed}/selected_pareto_solution_seed_{seed}.csv",
    ])

lgg_manifest = require_files(LGG_DIR, lgg_required, "TCGA-LGG")
lgg_manifest.to_csv(INPUT_VALIDATION_DIR / "lgg_required_file_manifest.csv", index=False)

lgg_split = pd.read_csv(LGG_DIR / "patient_split_assignments.csv")
lgg_candidates = pd.read_csv(LGG_DIR / "NSGA_candidate_gene_pool.csv")
lgg_train = pd.read_csv(LGG_DIR / "training_candidate_expression_log2TPM1.csv.gz")
lgg_valid = pd.read_csv(LGG_DIR / "heldout_validation_candidate_expression_log2TPM1.csv.gz")
lgg_selected_summary = pd.read_csv(LGG_DIR / "tables" / "selected_pareto_solution_by_seed.csv")

with open(LGG_DIR / "analysis_configuration.json", "r", encoding="utf-8") as handle:
    lgg_original_config = json.load(handle)

# Strict consistency checks.
assert len(lgg_train) == 197, f"Expected 197 LGG training patients, found {len(lgg_train)}"
assert len(lgg_valid) == 85, f"Expected 85 LGG held-out patients, found {len(lgg_valid)}"
assert len(lgg_candidates) == 3000, f"Expected 3,000 LGG candidate genes, found {len(lgg_candidates)}"
assert lgg_train.columns[0] == "case_submitter_id"
assert lgg_valid.columns[0] == "case_submitter_id"
assert lgg_train.columns[1:].tolist() == lgg_candidates["gene_id"].tolist()
assert lgg_valid.columns[1:].tolist() == lgg_candidates["gene_id"].tolist()

lgg_gene_to_symbol = dict(zip(lgg_candidates["gene_id"], lgg_candidates["gene_name"]))
lgg_candidate_genes = lgg_candidates["gene_id"].tolist()

lgg_panels = {}
for seed in LGG_SEEDS:
    panel_df = pd.read_csv(LGG_DIR / "nsga_multiseed_runs" / f"seed_{seed}" / f"selected_genes_seed_{seed}.csv")
    lgg_panels[seed] = panel_df["gene_id"].astype(str).tolist()
    if len(lgg_panels[seed]) != 50:
        raise RuntimeError(f"Seed {seed} selected {len(lgg_panels[seed])} genes, expected 50.")
    if not set(lgg_panels[seed]).issubset(set(lgg_candidate_genes)):
        raise RuntimeError(f"Seed {seed} contains genes outside the fixed 3,000-gene candidate pool.")

train_ids = set(lgg_train["case_submitter_id"])
valid_ids = set(lgg_valid["case_submitter_id"])
assert train_ids.isdisjoint(valid_ids)
assert train_ids == set(lgg_split.loc[lgg_split["data_split"] == "training", "case_submitter_id"])
assert valid_ids == set(lgg_split.loc[lgg_split["data_split"] == "heldout_validation", "case_submitter_id"])

lgg_input_summary = pd.DataFrame([{
    "training_patients": len(lgg_train),
    "heldout_validation_patients": len(lgg_valid),
    "candidate_genes": len(lgg_candidate_genes),
    "evolutionary_seeds": ";".join(map(str, LGG_SEEDS)),
    "selected_genes_per_seed": ";".join(str(len(lgg_panels[s])) for s in LGG_SEEDS),
    "unique_genes_across_five_selected_panels": len(set().union(*map(set, lgg_panels.values()))),
}])
lgg_input_summary.to_csv(INPUT_VALIDATION_DIR / "lgg_input_summary.csv", index=False)

display(lgg_input_summary)
print("LGG input validation passed.")

## 5. Validate and load the TCGA kidney TPM/CPM inputs

The kidney comparison uses the same fixed patients and split, but two separately processed expression representations. Full TPM and CPM matrices are retained so genes selected by one branch can still be evaluated in the other representation.

In [ ]:
KIDNEY_PROCESSED_DIR = resolve_root(KIDNEY_PROCESSED_REQUESTED, "tcga_kidney_sample_metadata_with_split.csv")
KIDNEY_MODELING_DIR = resolve_root(KIDNEY_MODELING_REQUESTED, "primary_tpm_vs_sensitivity_cpm_comparison.csv")

kidney_required_processed = [
    "tcga_kidney_sample_metadata_with_split.csv",
    "primary_log2_tpm_training.csv",
    "primary_log2_tpm_heldout_validation.csv",
    "sensitivity_log2_cpm_training.csv",
    "sensitivity_log2_cpm_heldout_validation.csv",
    "primary_log2_tpm_selected_genes.csv",
    "sensitivity_log2_cpm_selected_genes.csv",
    "tcga_kidney_tpm.csv.gz",
    "tcga_kidney_cpm.csv.gz",
    "tcga_kidney_gene_annotation.csv",
]
kidney_required_modeling = [
    "primary_log2_tpm/nsga_ii_final_selected_genes.csv",
    "sensitivity_log2_cpm/nsga_ii_final_selected_genes.csv",
    "primary_tpm_vs_sensitivity_cpm_comparison.csv",
    "nsga_selected_gene_overlap_summary.csv",
]

kidney_manifest = pd.concat([
    require_files(KIDNEY_PROCESSED_DIR, kidney_required_processed, "TCGA-Kidney processed"),
    require_files(KIDNEY_MODELING_DIR, kidney_required_modeling, "TCGA-Kidney modeling"),
], ignore_index=True)
kidney_manifest.to_csv(INPUT_VALIDATION_DIR / "kidney_required_file_manifest.csv", index=False)

kidney_metadata = pd.read_csv(KIDNEY_PROCESSED_DIR / "tcga_kidney_sample_metadata_with_split.csv")
kidney_tpm_candidates_df = pd.read_csv(KIDNEY_PROCESSED_DIR / "primary_log2_tpm_selected_genes.csv")
kidney_cpm_candidates_df = pd.read_csv(KIDNEY_PROCESSED_DIR / "sensitivity_log2_cpm_selected_genes.csv")
kidney_annotation = pd.read_csv(KIDNEY_PROCESSED_DIR / "tcga_kidney_gene_annotation.csv")
kidney_tpm_panel_df = pd.read_csv(KIDNEY_MODELING_DIR / "primary_log2_tpm" / "nsga_ii_final_selected_genes.csv")
kidney_cpm_panel_df = pd.read_csv(KIDNEY_MODELING_DIR / "sensitivity_log2_cpm" / "nsga_ii_final_selected_genes.csv")
kidney_performance = pd.read_csv(KIDNEY_MODELING_DIR / "primary_tpm_vs_sensitivity_cpm_comparison.csv")

kidney_tpm_candidates = kidney_tpm_candidates_df.loc[
    kidney_tpm_candidates_df["selected_for_nsga_ii_input"].astype(bool), "gene_id"
].astype(str).tolist()
kidney_cpm_candidates = kidney_cpm_candidates_df.loc[
    kidney_cpm_candidates_df["selected_for_nsga_ii_input"].astype(bool), "gene_id"
].astype(str).tolist()
kidney_tpm_panel = kidney_tpm_panel_df["gene_id"].astype(str).tolist()
kidney_cpm_panel = kidney_cpm_panel_df["gene_id"].astype(str).tolist()
kidney_gene_to_symbol = dict(zip(kidney_annotation["gene_id"], kidney_annotation["gene_name"]))

assert len(kidney_tpm_candidates) == 3000
assert len(kidney_cpm_candidates) == 3000
assert len(kidney_tpm_panel) == 50
assert len(kidney_cpm_panel) == 50
assert set(kidney_tpm_panel).issubset(set(kidney_tpm_candidates))
assert set(kidney_cpm_panel).issubset(set(kidney_cpm_candidates))

kidney_train_ids = kidney_metadata.loc[kidney_metadata["data_split"] == "training", "case_submitter_id"].astype(str).tolist()
kidney_valid_ids = kidney_metadata.loc[kidney_metadata["data_split"] == "validation", "case_submitter_id"].astype(str).tolist()
assert len(kidney_train_ids) == 622
assert len(kidney_valid_ids) == 267

kidney_input_summary = pd.DataFrame([{
    "training_patients": len(kidney_train_ids),
    "heldout_validation_patients": len(kidney_valid_ids),
    "TPM_candidate_genes": len(kidney_tpm_candidates),
    "CPM_candidate_genes": len(kidney_cpm_candidates),
    "candidate_pool_overlap": len(set(kidney_tpm_candidates) & set(kidney_cpm_candidates)),
    "TPM_selected_panel_genes": len(kidney_tpm_panel),
    "CPM_selected_panel_genes": len(kidney_cpm_panel),
    "selected_panel_overlap": len(set(kidney_tpm_panel) & set(kidney_cpm_panel)),
}])
kidney_input_summary.to_csv(INPUT_VALIDATION_DIR / "kidney_input_summary.csv", index=False)

display(kidney_input_summary)
print("Kidney input validation passed.")

## 6. Stability and correlation-matching functions

For a binary selection matrix with $M$ runs and $d$ candidate features, the Nogueira estimator is

\[
\widehat{\Phi}
=
1-
\frac{\frac{1}{d}\sum_{f=1}^{d}s_f^2}
{\frac{\bar{k}}{d}\left(1-\frac{\bar{k}}{d}\right)},
\qquad
s_f^2=\frac{M}{M-1}\widehat p_f(1-\widehat p_f).
\]

The proposed correlation-adjusted matching analysis is deliberately kept separate from this established estimator. For two panels, a bipartite graph is formed at threshold $\tau$ using edges with $|r|\ge\tau$. One-to-one matching is solved with a cardinality-first assignment so one gene cannot be counted as a substitute for several genes in the other panel. Two descriptive measures are reported:

- **matched coverage:** $m/\min(k_A,k_B)$;
- **correlation-adjusted Jaccard:** $m/(k_A+k_B-m)$.

When matches are exact identities only, the second expression reduces to ordinary Jaccard similarity.

In [ ]:
def selection_matrix(panels, universe):
    universe = list(universe)
    lookup = {g: j for j, g in enumerate(universe)}
    labels = list(panels.keys())
    Z = np.zeros((len(labels), len(universe)), dtype=np.uint8)
    for i, label in enumerate(labels):
        for gene in panels[label]:
            if gene not in lookup:
                raise KeyError(f"Gene {gene} is not in the supplied feature universe.")
            Z[i, lookup[gene]] = 1
    return Z, labels


def nogueira_stability(Z):
    Z = np.asarray(Z, dtype=float)
    M, d = Z.shape
    if M < 2:
        raise ValueError("Nogueira stability requires at least two selection runs.")
    p_hat = Z.mean(axis=0)
    sample_variance = (M / (M - 1.0)) * p_hat * (1.0 - p_hat)
    k_bar = Z.sum(axis=1).mean()
    denominator = (k_bar / d) * (1.0 - k_bar / d)
    if denominator <= 0:
        return np.nan
    return float(1.0 - sample_variance.mean() / denominator)


def jaccard(a, b):
    a, b = set(a), set(b)
    union = a | b
    return len(a & b) / len(union) if union else np.nan


def kuncheva_fixed_k(a, b, d):
    a, b = set(a), set(b)
    if len(a) != len(b):
        return np.nan
    k = len(a)
    expected = (k * k) / d
    denominator = k - expected
    return (len(a & b) - expected) / denominator if denominator else np.nan


def pairwise_exact_stability(panels, universe_size):
    rows = []
    for a, b in combinations(panels.keys(), 2):
        A, B = panels[a], panels[b]
        inter = len(set(A) & set(B))
        union = len(set(A) | set(B))
        rows.append({
            "panel_a": a,
            "panel_b": b,
            "genes_a": len(A),
            "genes_b": len(B),
            "intersection": inter,
            "union": union,
            "jaccard": inter / union if union else np.nan,
            "kuncheva_fixed_k": kuncheva_fixed_k(A, B, universe_size),
        })
    return pd.DataFrame(rows)


def standardize_columns(X):
    X = np.asarray(X, dtype=np.float32)
    means = X.mean(axis=0, dtype=np.float64)
    centered = X - means.astype(np.float32)
    std = centered.std(axis=0, ddof=1, dtype=np.float64)
    if np.any(std <= 0):
        bad = int(np.sum(std <= 0))
        raise RuntimeError(f"{bad} columns have zero variance and cannot be correlated.")
    return centered / std.astype(np.float32)


def correlation_matrix(X):
    Z = standardize_columns(X)
    corr = (Z.T @ Z) / np.float32(Z.shape[0] - 1)
    return np.clip(corr, -1.0, 1.0).astype(np.float32)


def cross_correlation_matrix(X_a, X_b):
    Za = standardize_columns(X_a)
    Zb = standardize_columns(X_b)
    corr = (Za.T @ Zb) / np.float32(Za.shape[0] - 1)
    return np.clip(corr, -1.0, 1.0).astype(np.float32)


def direct_pair_correlations(X_df, gene_pairs):
    rows = []
    for gene_a, gene_b in gene_pairs:
        a = X_df[gene_a].to_numpy(dtype=float)
        b = X_df[gene_b].to_numpy(dtype=float)
        if gene_a == gene_b:
            r = 1.0
        elif np.std(a, ddof=1) == 0 or np.std(b, ddof=1) == 0:
            r = np.nan
        else:
            r = float(np.corrcoef(a, b)[0, 1])
        rows.append(r)
    return np.asarray(rows, dtype=float)


def cardinality_first_match(abs_corr_submatrix, threshold):
    """
    One-to-one maximum matching with cardinality prioritized before correlation strength.
    A large constant makes any additional eligible edge more valuable than any possible
    change in the sum of correlation weights.
    """
    M = np.asarray(abs_corr_submatrix, dtype=float)
    eligible = np.isfinite(M) & (M >= float(threshold))
    if not eligible.any():
        return np.array([], dtype=int), np.array([], dtype=int), np.array([], dtype=float)
    BIG = 10_000.0
    score = np.where(eligible, BIG + M, 0.0)
    row_ind, col_ind = linear_sum_assignment(-score)
    keep = eligible[row_ind, col_ind]
    row_ind = row_ind[keep]
    col_ind = col_ind[keep]
    weights = M[row_ind, col_ind]
    return row_ind, col_ind, weights


def matching_summary(abs_corr_submatrix, threshold, k_a=None, k_b=None):
    M = np.asarray(abs_corr_submatrix, dtype=float)
    if k_a is None: k_a = M.shape[0]
    if k_b is None: k_b = M.shape[1]
    row_ind, col_ind, weights = cardinality_first_match(M, threshold)
    m = int(len(weights))
    denom_j = k_a + k_b - m
    return {
        "matched_count": m,
        "matched_coverage_smaller_panel": m / min(k_a, k_b) if min(k_a, k_b) else np.nan,
        "correlation_adjusted_jaccard": m / denom_j if denom_j else np.nan,
        "mean_abs_correlation_of_matches": float(np.mean(weights)) if m else np.nan,
        "median_abs_correlation_of_matches": float(np.median(weights)) if m else np.nan,
    }


def panel_match(panel_a, panel_b, gene_to_index_a, gene_to_index_b, abs_corr_matrix, threshold):
    ia = [gene_to_index_a[g] for g in panel_a]
    ib = [gene_to_index_b[g] for g in panel_b]
    sub = np.asarray(abs_corr_matrix[np.ix_(ia, ib)], dtype=float)
    rows, cols, weights = cardinality_first_match(sub, threshold)
    summary = matching_summary(sub, threshold, len(panel_a), len(panel_b))
    pairs = pd.DataFrame({
        "gene_a": [panel_a[i] for i in rows],
        "gene_b": [panel_b[j] for j in cols],
        "abs_correlation": weights,
    })
    if not pairs.empty:
        pairs["exact_gene_identity"] = pairs["gene_a"] == pairs["gene_b"]
    return summary, pairs


def maximum_cardinality_match_count(abs_corr_submatrix, threshold):
    """Return only the maximum eligible one-to-one matching cardinality.

    This sparse bipartite implementation is used for large null simulations where
    the identity of the matched pairs is irrelevant. It is mathematically
    consistent with the cardinality-first matching used for observed panels.
    """
    M = np.asarray(abs_corr_submatrix, dtype=float)
    eligible = np.isfinite(M) & (M >= float(threshold))
    if not eligible.any():
        return 0
    graph = csr_matrix(eligible.astype(np.int8, copy=False))
    match = maximum_bipartite_matching(graph, perm_type="column")
    return int(np.sum(match >= 0))


def corr_adjusted_jaccard_from_match_count(matched_count, k_a, k_b):
    denominator = int(k_a) + int(k_b) - int(matched_count)
    return float(matched_count / denominator) if denominator else np.nan


def chance_corrected_similarity(observed, expected_null):
    """Chance-correct a similarity whose theoretical upper bound is one.

    Values >0 exceed the null expectation, 0 equals the null expectation, and
    negative values fall below the null expectation. This is an analysis-defined
    chance correction, not a claim that the raw correlation-adjusted Jaccard is an
    established chance-corrected stability estimator.
    """
    observed = np.asarray(observed, dtype=float)
    expected_null = np.asarray(expected_null, dtype=float)
    denominator = 1.0 - expected_null
    return np.where(np.abs(denominator) > 1e-12,
                    (observed - expected_null) / denominator,
                    np.nan)


## 7. LGG exact gene stability across the five evolutionary seeds

In [ ]:
Z_lgg, lgg_seed_order = selection_matrix(lgg_panels, lgg_candidate_genes)
lgg_exact_pairwise = pairwise_exact_stability(lgg_panels, universe_size=len(lgg_candidate_genes))
lgg_exact_pairwise.to_csv(EXACT_DIR / "LGG_pairwise_exact_stability.csv", index=False)

recurrence = Z_lgg.sum(axis=0)
lgg_recurrence = lgg_candidates[["gene_id", "gene_name"]].copy()
lgg_recurrence["selected_seed_count"] = recurrence
lgg_recurrence = lgg_recurrence.loc[lgg_recurrence["selected_seed_count"] > 0].sort_values(
    ["selected_seed_count", "gene_name"], ascending=[False, True], kind="stable"
)
lgg_recurrence.to_csv(EXACT_DIR / "LGG_selected_gene_recurrence_across_seeds.csv", index=False)

recurrence_distribution = (
    lgg_recurrence["selected_seed_count"].value_counts().sort_index().rename_axis("seeds_selected_in")
    .reset_index(name="number_of_unique_genes")
)
recurrence_distribution.to_csv(EXACT_DIR / "LGG_gene_recurrence_distribution.csv", index=False)

exact_summary = pd.DataFrame([{
    "selection_runs": len(lgg_panels),
    "candidate_universe": len(lgg_candidate_genes),
    "genes_per_selected_panel": 50,
    "total_selection_slots": int(Z_lgg.sum()),
    "unique_selected_genes": int((recurrence > 0).sum()),
    "mean_pairwise_overlap": lgg_exact_pairwise["intersection"].mean(),
    "mean_pairwise_jaccard": lgg_exact_pairwise["jaccard"].mean(),
    "mean_pairwise_kuncheva": lgg_exact_pairwise["kuncheva_fixed_k"].mean(),
}])
exact_summary.to_csv(EXACT_DIR / "LGG_exact_stability_summary.csv", index=False)

display(exact_summary)
display(lgg_exact_pairwise)

# Jaccard heat map generated directly from the five saved panels.
labels = [str(s) for s in LGG_SEEDS]
J = np.eye(len(LGG_SEEDS), dtype=float)
for i, sa in enumerate(LGG_SEEDS):
    for j, sb in enumerate(LGG_SEEDS):
        J[i, j] = jaccard(lgg_panels[sa], lgg_panels[sb])

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(J, vmin=0, vmax=1, cmap="Blues")
ax.set_xticks(range(len(labels)), labels=labels)
ax.set_yticks(range(len(labels)), labels=labels)
ax.set_xlabel("Evolutionary seed")
ax.set_ylabel("Evolutionary seed")
ax.set_title("Exact Gene Jaccard Similarity Across LGG NSGA-II Seeds")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{J[i,j]:.3f}", ha="center", va="center",
                color="white" if J[i,j] > 0.5 else "black", fontsize=9)
fig.colorbar(im, ax=ax, label="Jaccard similarity")
fig.tight_layout()
save_figure(fig, "LGG_exact_Jaccard_heatmap", EXACT_DIR)
plt.show()
plt.close(fig)

## 8. LGG Nogueira stability estimator

The five LGG searches share the same 3,000-gene feature universe and all selected 50 genes, so the Nogueira estimator is directly applicable to the evolutionary-seed perturbation.

In [ ]:
lgg_phi = nogueira_stability(Z_lgg)

lgg_nogueira = pd.DataFrame([{
    "perturbation": "evolutionary_seed",
    "selection_runs_M": Z_lgg.shape[0],
    "candidate_features_d": Z_lgg.shape[1],
    "mean_selected_features_k": Z_lgg.sum(axis=1).mean(),
    "nogueira_stability_phi": lgg_phi,
    "theoretical_finite_M_lower_bound": -1.0 / (Z_lgg.shape[0] - 1),
}])
lgg_nogueira.to_csv(NOGUEIRA_DIR / "LGG_Nogueira_stability_evolutionary_seeds.csv", index=False)
display(lgg_nogueira)

## 9. LGG training correlation matrix and correlation-adjusted panel similarity

The full 3,000 × 3,000 training correlation matrix is cached as a binary NumPy array for efficient null simulations. All correlations are calculated from the fixed 197-patient training cohort only.

In [ ]:
lgg_gene_to_index = {g: i for i, g in enumerate(lgg_candidate_genes)}
LGG_CORR_CACHE = CACHE_DIR / "LGG_training_candidate_correlation_float32.npy"

if LGG_CORR_CACHE.exists():
    print("Loading cached LGG training correlation matrix...")
    lgg_train_corr = np.load(LGG_CORR_CACHE, mmap_mode="r")
else:
    print("Computing 3,000 x 3,000 LGG training correlation matrix...")
    X = lgg_train[lgg_candidate_genes].to_numpy(dtype=np.float32)
    with threadpool_limits(limits=max(1, min(N_WORKERS, 10))):
        lgg_train_corr = correlation_matrix(X)
    np.save(LGG_CORR_CACHE, lgg_train_corr.astype(np.float32))
    lgg_train_corr = np.load(LGG_CORR_CACHE, mmap_mode="r")

lgg_abs_corr = np.abs(lgg_train_corr)

threshold_rows = []
primary_match_rows = []
for seed_a, seed_b in combinations(LGG_SEEDS, 2):
    panel_a = lgg_panels[seed_a]
    panel_b = lgg_panels[seed_b]
    for tau in CORRELATION_THRESHOLDS:
        summary, pairs = panel_match(
            panel_a, panel_b,
            lgg_gene_to_index, lgg_gene_to_index,
            lgg_abs_corr, tau
        )
        threshold_rows.append({
            "seed_a": seed_a, "seed_b": seed_b, "threshold": tau,
            "exact_jaccard": jaccard(panel_a, panel_b),
            **summary,
        })
        if np.isclose(tau, PRIMARY_CORRELATION_THRESHOLD) and not pairs.empty:
            pairs = pairs.copy()
            pairs.insert(0, "seed_b", seed_b)
            pairs.insert(0, "seed_a", seed_a)
            pairs["gene_name_a"] = pairs["gene_a"].map(lgg_gene_to_symbol)
            pairs["gene_name_b"] = pairs["gene_b"].map(lgg_gene_to_symbol)
            primary_match_rows.append(pairs)

lgg_corr_adjusted = pd.DataFrame(threshold_rows)
lgg_corr_adjusted.to_csv(CORR_DIR / "LGG_correlation_adjusted_similarity_by_threshold.csv", index=False)

lgg_primary_matches = pd.concat(primary_match_rows, ignore_index=True) if primary_match_rows else pd.DataFrame()
lgg_primary_matches.to_csv(CORR_DIR / f"LGG_one_to_one_matches_tau_{PRIMARY_CORRELATION_THRESHOLD:.2f}.csv", index=False)

summary_by_tau = lgg_corr_adjusted.groupby("threshold").agg(
    mean_exact_jaccard=("exact_jaccard", "mean"),
    mean_adjusted_jaccard=("correlation_adjusted_jaccard", "mean"),
    median_adjusted_jaccard=("correlation_adjusted_jaccard", "median"),
    mean_matched_coverage=("matched_coverage_smaller_panel", "mean"),
    mean_matched_count=("matched_count", "mean"),
).reset_index()
summary_by_tau.to_csv(CORR_DIR / "LGG_correlation_adjusted_similarity_threshold_summary.csv", index=False)
display(summary_by_tau)

fig, ax = plt.subplots(figsize=(7.5, 5.3))
for (sa, sb), group in lgg_corr_adjusted.groupby(["seed_a", "seed_b"]):
    ax.plot(group["threshold"], group["correlation_adjusted_jaccard"],
            marker="o", linewidth=1, alpha=0.35)
median_curve = lgg_corr_adjusted.groupby("threshold")["correlation_adjusted_jaccard"].median()
ax.plot(median_curve.index, median_curve.values, marker="o", linewidth=3,
        color=DISSERTATION_NAVY, label="Median across 10 seed pairs")
ax.axvline(PRIMARY_CORRELATION_THRESHOLD, linestyle="--", linewidth=1, color="black")
ax.set_xlabel("Absolute correlation threshold")
ax.set_ylabel("Correlation-adjusted Jaccard")
ax.set_title("LGG Correlation-Adjusted Panel Similarity Across Thresholds")
ax.set_ylim(0, 1.02)
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "LGG_correlation_adjusted_Jaccard_threshold_curve", CORR_DIR)
plt.show()
plt.close(fig)

## 10. LGG preservation of training-derived matches in held-out patients

Matching is performed **only in the training cohort** at the prespecified primary threshold. The same gene pairs are then evaluated in the 85 held-out patients without rematching.

In [ ]:
if lgg_primary_matches.empty:
    raise RuntimeError("No LGG training matches were found at the primary correlation threshold.")

pair_list = list(zip(lgg_primary_matches["gene_a"], lgg_primary_matches["gene_b"]))
validation_r = direct_pair_correlations(lgg_valid.set_index("case_submitter_id"), pair_list)

lgg_match_preservation = lgg_primary_matches.copy()
lgg_match_preservation["r_training"] = [
    float(lgg_train_corr[lgg_gene_to_index[a], lgg_gene_to_index[b]])
    for a, b in pair_list
]
lgg_match_preservation["r_heldout_validation"] = validation_r
lgg_match_preservation["abs_r_training"] = np.abs(lgg_match_preservation["r_training"])
lgg_match_preservation["abs_r_heldout_validation"] = np.abs(lgg_match_preservation["r_heldout_validation"])
lgg_match_preservation["preserved_at_same_threshold"] = (
    lgg_match_preservation["abs_r_heldout_validation"] >= PRIMARY_CORRELATION_THRESHOLD
)
lgg_match_preservation["change_in_abs_r_validation_minus_training"] = (
    lgg_match_preservation["abs_r_heldout_validation"] - lgg_match_preservation["abs_r_training"]
)
lgg_match_preservation.to_csv(
    PRESERVATION_DIR / f"LGG_training_matches_heldout_preservation_tau_{PRIMARY_CORRELATION_THRESHOLD:.2f}.csv",
    index=False,
)

lgg_preservation_summary = lgg_match_preservation.groupby(["seed_a", "seed_b"]).agg(
    training_matches=("gene_a", "size"),
    exact_identity_matches=("exact_gene_identity", "sum"),
    preserved_matches=("preserved_at_same_threshold", "sum"),
    preservation_fraction=("preserved_at_same_threshold", "mean"),
    median_abs_r_training=("abs_r_training", "median"),
    median_abs_r_validation=("abs_r_heldout_validation", "median"),
    median_change_abs_r=("change_in_abs_r_validation_minus_training", "median"),
).reset_index()
lgg_preservation_summary.to_csv(PRESERVATION_DIR / "LGG_match_preservation_summary_by_seed_pair.csv", index=False)
display(lgg_preservation_summary)

fig, ax = plt.subplots(figsize=(6.3, 6.0))
ax.scatter(lgg_match_preservation["abs_r_training"],
           lgg_match_preservation["abs_r_heldout_validation"], alpha=0.65)
ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
ax.axhline(PRIMARY_CORRELATION_THRESHOLD, linestyle=":", color="black", linewidth=1)
ax.axvline(PRIMARY_CORRELATION_THRESHOLD, linestyle=":", color="black", linewidth=1)
ax.set_xlim(0.45, 1.02)
ax.set_ylim(0.45, 1.02)
ax.set_xlabel("|r| in 197 training patients")
ax.set_ylabel("|r| in 85 held-out patients")
ax.set_title("LGG Preservation of Training-Derived Gene Matches")
fig.tight_layout()
save_figure(fig, "LGG_training_vs_heldout_matched_gene_correlations", PRESERVATION_DIR)
plt.show()
plt.close(fig)

## 11. LGG pairwise null model and chance-corrected correlation-adjusted stability

The original pairwise null analysis is retained at the prespecified primary threshold. This revision additionally generates a **single random-panel null curve across all correlation thresholds**. The mean null similarity at each threshold is used to calculate an analysis-defined chance-corrected correlation-adjusted stability score:

\[
S_{\mathrm{CC}}(	au)=
rac{S_{\mathrm{obs}}(	au)-E\{S_{\mathrm{null}}(	au)\}}{1-E\{S_{\mathrm{null}}(	au)\}}.
\]

Thus, zero indicates similarity equal to random 50-gene panels drawn from the same 3,000-gene candidate universe; positive values exceed the null expectation; and negative values fall below it.


In [ ]:

def _null_lgg_chunk(n_rep, seed, abs_corr_matrix, threshold, d=3000, k=50):
    rng = np.random.default_rng(seed)
    out = np.empty((n_rep, 5), dtype=float)
    for i in range(n_rep):
        a = rng.choice(d, size=k, replace=False)
        b = rng.choice(d, size=k, replace=False)
        exact = len(np.intersect1d(a, b, assume_unique=False))
        sub = np.asarray(abs_corr_matrix[np.ix_(a, b)], dtype=float)
        m = maximum_cardinality_match_count(sub, threshold)
        out[i] = [
            exact,
            exact / (2*k-exact),
            m,
            m / k,
            corr_adjusted_jaccard_from_match_count(m, k, k),
        ]
    return out


def simulate_lgg_null(n_replicates, threshold):
    cache = NULL_DIR / f"LGG_null_n{n_replicates}_tau{threshold:.2f}.csv"
    if cache.exists():
        existing = pd.read_csv(cache)
        if len(existing) == n_replicates:
            print("Loading cached LGG null distribution:", cache.name)
            return existing
    counts = np.full(N_WORKERS, n_replicates // N_WORKERS, dtype=int)
    counts[: n_replicates % N_WORKERS] += 1
    jobs = [(int(n), RANDOM_SEED + 1000 + j) for j, n in enumerate(counts) if n > 0]
    print(f"Simulating {n_replicates:,} LGG null panel pairs using {len(jobs)} workers...")
    parts = Parallel(
        n_jobs=N_WORKERS, backend=JOBLIB_BACKEND,
        temp_folder=str(JOBLIB_TEMP_DIR), max_nbytes="10M", mmap_mode="r"
    )(
        delayed(_null_lgg_chunk)(n, seed, lgg_abs_corr, threshold)
        for n, seed in jobs
    )
    arr = np.vstack(parts)
    df = pd.DataFrame(arr, columns=[
        "exact_overlap", "exact_jaccard", "matched_count",
        "matched_coverage", "correlation_adjusted_jaccard"
    ])
    df.insert(0, "null_replicate", np.arange(1, len(df)+1))
    df.to_csv(cache, index=False)
    return df


# Primary-threshold pairwise null retained from revision 1.
lgg_null = simulate_lgg_null(N_NULL_REPLICATES, PRIMARY_CORRELATION_THRESHOLD)

observed_primary = lgg_corr_adjusted.loc[
    np.isclose(lgg_corr_adjusted["threshold"], PRIMARY_CORRELATION_THRESHOLD)
].copy()

null_test_rows = []
for _, row in observed_primary.iterrows():
    p_exact = (1 + np.sum(lgg_null["exact_jaccard"] >= row["exact_jaccard"])) / (len(lgg_null) + 1)
    p_corr = (1 + np.sum(lgg_null["correlation_adjusted_jaccard"] >= row["correlation_adjusted_jaccard"])) / (len(lgg_null) + 1)
    null_test_rows.append({
        "seed_a": int(row["seed_a"]),
        "seed_b": int(row["seed_b"]),
        "observed_exact_jaccard": row["exact_jaccard"],
        "empirical_p_exact_jaccard_ge_observed": p_exact,
        "observed_correlation_adjusted_jaccard": row["correlation_adjusted_jaccard"],
        "empirical_p_corr_adjusted_jaccard_ge_observed": p_corr,
        "null_replicates": len(lgg_null),
    })

lgg_null_tests = pd.DataFrame(null_test_rows)
lgg_null_tests.to_csv(NULL_DIR / "LGG_observed_vs_random_panel_null_tests.csv", index=False)
display(lgg_null_tests)

fig, ax = plt.subplots(figsize=(7.2, 5.0))
ax.hist(lgg_null["correlation_adjusted_jaccard"], bins=35, alpha=0.75)
for value in observed_primary["correlation_adjusted_jaccard"]:
    ax.axvline(value, linewidth=1, alpha=0.55, color=DISSERTATION_NAVY)
ax.set_xlabel(f"Correlation-adjusted Jaccard (|r| ≥ {PRIMARY_CORRELATION_THRESHOLD:.2f})")
ax.set_ylabel("Null panel-pair count")
ax.set_title("LGG Random-Panel Null Distribution")
fig.tight_layout()
save_figure(fig, "LGG_null_distribution_corr_adjusted_Jaccard", NULL_DIR)
plt.show()
plt.close(fig)


# ------------------------------------------------------------
# Null expectation across the complete correlation-threshold curve.
# ------------------------------------------------------------
def _null_lgg_threshold_curve_chunk(n_rep, seed, abs_corr_matrix, thresholds, d=3000, k=50):
    rng = np.random.default_rng(seed)
    thresholds = np.asarray(thresholds, dtype=float)
    out = np.empty((n_rep, len(thresholds)), dtype=np.float32)
    for i in range(n_rep):
        a = rng.choice(d, size=k, replace=False)
        b = rng.choice(d, size=k, replace=False)
        sub = np.asarray(abs_corr_matrix[np.ix_(a, b)], dtype=float)
        for j, tau in enumerate(thresholds):
            m = maximum_cardinality_match_count(sub, tau)
            out[i, j] = corr_adjusted_jaccard_from_match_count(m, k, k)
    return out


def simulate_lgg_null_threshold_curve(n_replicates, thresholds):
    thresholds = [float(t) for t in thresholds]
    cache = NULL_DIR / f"LGG_pairwise_null_threshold_curve_n{n_replicates}.csv"
    expected_cols = [f"tau_{t:.2f}" for t in thresholds]
    if cache.exists():
        existing = pd.read_csv(cache)
        if len(existing) == n_replicates and all(c in existing.columns for c in expected_cols):
            print("Loading cached LGG pairwise null threshold curve:", cache.name)
            return existing
    counts = np.full(N_WORKERS, n_replicates // N_WORKERS, dtype=int)
    counts[: n_replicates % N_WORKERS] += 1
    jobs = [(int(n), RANDOM_SEED + 2000 + j) for j, n in enumerate(counts) if n > 0]
    print(f"Simulating LGG null similarity across {len(thresholds)} thresholds and {n_replicates:,} panel pairs...")
    parts = Parallel(
        n_jobs=N_WORKERS, backend=JOBLIB_BACKEND,
        temp_folder=str(JOBLIB_TEMP_DIR), max_nbytes="10M", mmap_mode="r"
    )(
        delayed(_null_lgg_threshold_curve_chunk)(n, seed, lgg_abs_corr, thresholds)
        for n, seed in jobs
    )
    arr = np.vstack(parts)
    df = pd.DataFrame(arr, columns=expected_cols)
    df.insert(0, "null_replicate", np.arange(1, len(df)+1))
    df.to_csv(cache, index=False)
    return df


lgg_null_curve = simulate_lgg_null_threshold_curve(N_NULL_REPLICATES, CORRELATION_THRESHOLDS)

lgg_null_curve_summary_rows = []
for tau in CORRELATION_THRESHOLDS:
    values = lgg_null_curve[f"tau_{tau:.2f}"].to_numpy(dtype=float)
    lgg_null_curve_summary_rows.append({
        "threshold": tau,
        "null_mean_corr_adjusted_jaccard": float(np.mean(values)),
        "null_median_corr_adjusted_jaccard": float(np.median(values)),
        "null_q025_corr_adjusted_jaccard": float(np.quantile(values, 0.025)),
        "null_q975_corr_adjusted_jaccard": float(np.quantile(values, 0.975)),
        "null_replicates": len(values),
    })
lgg_null_curve_summary = pd.DataFrame(lgg_null_curve_summary_rows)
lgg_null_curve_summary.to_csv(NULL_DIR / "LGG_pairwise_null_threshold_curve_summary.csv", index=False)

lgg_corr_adjusted_chance = lgg_corr_adjusted.merge(
    lgg_null_curve_summary[["threshold", "null_mean_corr_adjusted_jaccard"]],
    on="threshold", how="left"
)
lgg_corr_adjusted_chance["chance_corrected_corr_adjusted_stability"] = chance_corrected_similarity(
    lgg_corr_adjusted_chance["correlation_adjusted_jaccard"],
    lgg_corr_adjusted_chance["null_mean_corr_adjusted_jaccard"],
)
lgg_corr_adjusted_chance.to_csv(
    CORR_DIR / "LGG_chance_corrected_correlation_adjusted_similarity_by_threshold.csv", index=False
)

lgg_chance_threshold_summary = lgg_corr_adjusted_chance.groupby("threshold").agg(
    observed_mean_corr_adjusted_jaccard=("correlation_adjusted_jaccard", "mean"),
    observed_median_corr_adjusted_jaccard=("correlation_adjusted_jaccard", "median"),
    null_mean_corr_adjusted_jaccard=("null_mean_corr_adjusted_jaccard", "first"),
    mean_chance_corrected_corr_adjusted_stability=("chance_corrected_corr_adjusted_stability", "mean"),
    median_chance_corrected_corr_adjusted_stability=("chance_corrected_corr_adjusted_stability", "median"),
).reset_index()
lgg_chance_threshold_summary.to_csv(
    CORR_DIR / "LGG_chance_corrected_correlation_adjusted_threshold_summary.csv", index=False
)
display(lgg_chance_threshold_summary)

fig, ax = plt.subplots(figsize=(7.5, 5.2))
ax.plot(lgg_chance_threshold_summary["threshold"],
        lgg_chance_threshold_summary["observed_mean_corr_adjusted_jaccard"],
        marker="o", linewidth=2.5, label="Observed mean across 10 seed pairs")
ax.plot(lgg_chance_threshold_summary["threshold"],
        lgg_chance_threshold_summary["null_mean_corr_adjusted_jaccard"],
        marker="o", linewidth=2.5, label="Random-panel null mean")
ax.axvline(PRIMARY_CORRELATION_THRESHOLD, linestyle="--", linewidth=1, color="black")
ax.set_xlabel("Absolute correlation threshold")
ax.set_ylabel("Correlation-adjusted Jaccard")
ax.set_ylim(0, 1.02)
ax.set_title("LGG Observed Versus Null Correlation-Adjusted Similarity")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "LGG_observed_vs_null_corr_adjusted_threshold_curve", CORR_DIR)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.5, 5.2))
ax.plot(lgg_chance_threshold_summary["threshold"],
        lgg_chance_threshold_summary["mean_chance_corrected_corr_adjusted_stability"],
        marker="o", linewidth=2.5)
ax.axhline(0.0, linestyle="--", linewidth=1, color="black")
ax.axvline(PRIMARY_CORRELATION_THRESHOLD, linestyle="--", linewidth=1, color="black")
ax.set_xlabel("Absolute correlation threshold")
ax.set_ylabel("Chance-corrected correlation-adjusted stability")
ax.set_title("LGG Chance-Corrected Correlation-Adjusted Stability")
fig.tight_layout()
save_figure(fig, "LGG_chance_corrected_corr_adjusted_threshold_curve", CORR_DIR)
plt.show()
plt.close(fig)


## 11A. Global five-panel LGG null experiment and multi-seed component-structure testing

The pairwise null above answers whether one observed panel pair is unusually similar. It does **not** reproduce the complete five-run experimental design. This global null therefore draws **five independent 50-gene panels** from the same 3,000-gene candidate universe on each replicate and recalculates the same cross-panel structure.

For every replicate, the notebook records mean exact Jaccard across all ten panel pairs, mean correlation-adjusted Jaccard, Nogueira stability, total cross-seed network edges, the number and size of nontrivial correlation components, and how many correlated components contain genes representing four or all five simulated panels. Network edges use the same dissertation rule as the observed analysis: genes are linked only when their panel-membership sets are disjoint and training $|r|$ meets the primary threshold.


In [ ]:

def _nogueira_from_index_panels(panel_indices, d):
    M = len(panel_indices)
    counts = np.bincount(np.concatenate(panel_indices), minlength=d).astype(float)
    p_hat = counts / M
    sample_variance = (M / (M - 1.0)) * p_hat * (1.0 - p_hat)
    k_bar = np.mean([len(x) for x in panel_indices])
    denominator = (k_bar / d) * (1.0 - k_bar / d)
    return float(1.0 - sample_variance.mean() / denominator) if denominator > 0 else np.nan


def _cross_panel_component_metrics_from_indices(panel_indices, abs_corr_matrix, threshold, d):
    """Efficiently reproduce the observed disjoint-membership network definition."""
    membership = np.zeros(d, dtype=np.uint8)
    for panel_number, panel in enumerate(panel_indices):
        membership[np.asarray(panel, dtype=int)] |= np.uint8(1 << panel_number)

    selected = np.flatnonzero(membership)
    masks = membership[selected]
    n = len(selected)
    if n == 0:
        return {
            "unique_selected_genes": 0, "network_edge_count": 0,
            "nontrivial_components_ge2": 0, "large_components_ge4": 0,
            "largest_component_size": 0, "genes_in_nontrivial_components": 0,
            "nontrivial_components_seeds_ge4": 0, "nontrivial_components_all5": 0,
            "large_components_seeds_ge4": 0, "large_components_all5": 0,
            "largest_component_seed_coverage": 0,
        }

    sub = np.asarray(abs_corr_matrix[np.ix_(selected, selected)], dtype=float)
    disjoint = (masks[:, None] & masks[None, :]) == 0
    eligible = (sub >= float(threshold)) & disjoint
    np.fill_diagonal(eligible, False)
    rows, cols = np.where(np.triu(eligible, k=1))

    parent = np.arange(n, dtype=int)
    rank = np.zeros(n, dtype=np.uint8)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(int(a)), find(int(b))
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        if rank[ra] == rank[rb]:
            rank[ra] += 1

    for a, b in zip(rows, cols):
        union(a, b)

    roots = np.asarray([find(i) for i in range(n)], dtype=int)
    component_sizes = []
    component_seed_counts = []
    for root in np.unique(roots):
        member_idx = np.flatnonzero(roots == root)
        component_sizes.append(len(member_idx))
        combined_mask = int(np.bitwise_or.reduce(masks[member_idx]))
        component_seed_counts.append(combined_mask.bit_count())

    component_sizes = np.asarray(component_sizes, dtype=int)
    component_seed_counts = np.asarray(component_seed_counts, dtype=int)
    nontrivial = component_sizes >= 2
    large = component_sizes >= NETWORK_MIN_COMPONENT_SIZE

    return {
        "unique_selected_genes": int(n),
        "network_edge_count": int(len(rows)),
        "nontrivial_components_ge2": int(np.sum(nontrivial)),
        "large_components_ge4": int(np.sum(large)),
        "largest_component_size": int(component_sizes.max(initial=0)),
        "genes_in_nontrivial_components": int(component_sizes[nontrivial].sum()) if nontrivial.any() else 0,
        "nontrivial_components_seeds_ge4": int(np.sum(nontrivial & (component_seed_counts >= 4))),
        "nontrivial_components_all5": int(np.sum(nontrivial & (component_seed_counts == 5))),
        "large_components_seeds_ge4": int(np.sum(large & (component_seed_counts >= 4))),
        "large_components_all5": int(np.sum(large & (component_seed_counts == 5))),
        "largest_component_seed_coverage": int(component_seed_counts[np.argmax(component_sizes)]) if len(component_sizes) else 0,
    }


def global_five_panel_metrics(panel_indices, abs_corr_matrix, threshold, d=3000):
    pair_exact = []
    pair_corr = []
    pair_match_counts = []
    for ia, ib in combinations(range(len(panel_indices)), 2):
        a = np.asarray(panel_indices[ia], dtype=int)
        b = np.asarray(panel_indices[ib], dtype=int)
        exact = len(np.intersect1d(a, b, assume_unique=False))
        pair_exact.append(exact / (len(a) + len(b) - exact))
        sub = np.asarray(abs_corr_matrix[np.ix_(a, b)], dtype=float)
        m = maximum_cardinality_match_count(sub, threshold)
        pair_match_counts.append(m)
        pair_corr.append(corr_adjusted_jaccard_from_match_count(m, len(a), len(b)))

    metrics = {
        "mean_exact_jaccard": float(np.mean(pair_exact)),
        "mean_correlation_adjusted_jaccard": float(np.mean(pair_corr)),
        "mean_matched_count": float(np.mean(pair_match_counts)),
        "nogueira_stability": _nogueira_from_index_panels(panel_indices, d),
    }
    metrics.update(_cross_panel_component_metrics_from_indices(
        panel_indices, abs_corr_matrix, threshold, d
    ))
    return metrics


GLOBAL_NULL_COLUMNS = [
    "mean_exact_jaccard", "mean_correlation_adjusted_jaccard", "mean_matched_count",
    "nogueira_stability", "unique_selected_genes", "network_edge_count",
    "nontrivial_components_ge2", "large_components_ge4", "largest_component_size",
    "genes_in_nontrivial_components", "nontrivial_components_seeds_ge4",
    "nontrivial_components_all5", "large_components_seeds_ge4",
    "large_components_all5", "largest_component_seed_coverage",
]


def _global_lgg_null_chunk(n_rep, seed, abs_corr_matrix, threshold, d=3000, k=50, m_panels=5):
    rng = np.random.default_rng(seed)
    out = np.empty((n_rep, len(GLOBAL_NULL_COLUMNS)), dtype=float)
    for i in range(n_rep):
        panels = [rng.choice(d, size=k, replace=False) for _ in range(m_panels)]
        metrics = global_five_panel_metrics(panels, abs_corr_matrix, threshold, d=d)
        out[i] = [metrics[c] for c in GLOBAL_NULL_COLUMNS]
    return out


def simulate_lgg_global_five_panel_null(n_replicates, threshold):
    cache = NULL_DIR / f"LGG_global_five_panel_null_n{n_replicates}_tau{threshold:.2f}.csv"
    if cache.exists():
        existing = pd.read_csv(cache)
        if len(existing) == n_replicates and all(c in existing.columns for c in GLOBAL_NULL_COLUMNS):
            print("Loading cached LGG global five-panel null:", cache.name)
            return existing
    counts = np.full(N_WORKERS, n_replicates // N_WORKERS, dtype=int)
    counts[: n_replicates % N_WORKERS] += 1
    jobs = [(int(n), RANDOM_SEED + 5000 + j) for j, n in enumerate(counts) if n > 0]
    print(f"Simulating {n_replicates:,} complete five-panel LGG null experiments using {len(jobs)} workers...")
    parts = Parallel(
        n_jobs=N_WORKERS, backend=JOBLIB_BACKEND,
        temp_folder=str(JOBLIB_TEMP_DIR), max_nbytes="10M", mmap_mode="r"
    )(
        delayed(_global_lgg_null_chunk)(n, seed, lgg_abs_corr, threshold)
        for n, seed in jobs
    )
    arr = np.vstack(parts)
    df = pd.DataFrame(arr, columns=GLOBAL_NULL_COLUMNS)
    df.insert(0, "null_replicate", np.arange(1, len(df)+1))
    df.to_csv(cache, index=False)
    return df


observed_panel_indices = [
    np.asarray([lgg_gene_to_index[g] for g in lgg_panels[seed]], dtype=int)
    for seed in LGG_SEEDS
]
lgg_global_observed_metrics = global_five_panel_metrics(
    observed_panel_indices, lgg_abs_corr, PRIMARY_CORRELATION_THRESHOLD, d=len(lgg_candidate_genes)
)
lgg_global_null = simulate_lgg_global_five_panel_null(
    N_NULL_REPLICATES, PRIMARY_CORRELATION_THRESHOLD
)

# Add a chance-corrected global correlation-adjusted similarity.
global_null_corr_mean = float(lgg_global_null["mean_correlation_adjusted_jaccard"].mean())
lgg_global_observed_metrics["chance_corrected_mean_correlation_adjusted_stability"] = float(
    chance_corrected_similarity(
        lgg_global_observed_metrics["mean_correlation_adjusted_jaccard"],
        global_null_corr_mean,
    )
)

lgg_global_observed = pd.DataFrame([lgg_global_observed_metrics])
lgg_global_observed.to_csv(NULL_DIR / "LGG_global_five_panel_observed_metrics.csv", index=False)

# Report both empirical tails. Direction depends on the scientific question:
# high-tail for unusually strong stability/structure; low-tail for unusually weak stability.
global_test_rows = []
for metric in GLOBAL_NULL_COLUMNS:
    observed = float(lgg_global_observed_metrics[metric])
    null_values = lgg_global_null[metric].to_numpy(dtype=float)
    global_test_rows.append({
        "metric": metric,
        "observed": observed,
        "null_mean": float(np.mean(null_values)),
        "null_median": float(np.median(null_values)),
        "null_q025": float(np.quantile(null_values, 0.025)),
        "null_q975": float(np.quantile(null_values, 0.975)),
        "empirical_p_null_ge_observed": (1 + np.sum(null_values >= observed)) / (len(null_values) + 1),
        "empirical_p_null_le_observed": (1 + np.sum(null_values <= observed)) / (len(null_values) + 1),
        "null_replicates": len(null_values),
    })
lgg_global_null_tests = pd.DataFrame(global_test_rows)
lgg_global_null_tests.to_csv(NULL_DIR / "LGG_global_five_panel_observed_vs_null_tests.csv", index=False)

display(lgg_global_observed)
display(lgg_global_null_tests)

fig, ax = plt.subplots(figsize=(7.2, 5.0))
ax.hist(lgg_global_null["largest_component_size"], bins=np.arange(0.5, lgg_global_null["largest_component_size"].max()+2.5, 1), alpha=0.8)
ax.axvline(lgg_global_observed_metrics["largest_component_size"], linewidth=2.5, color=DISSERTATION_NAVY,
           label=f"Observed = {lgg_global_observed_metrics['largest_component_size']}")
ax.set_xlabel("Largest cross-panel correlation component size")
ax.set_ylabel("Five-panel null replicate count")
ax.set_title("LGG Global Five-Panel Null: Largest Component")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "LGG_global_null_largest_component_size", NULL_DIR)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.2, 5.0))
metric = "large_components_seeds_ge4"
values = lgg_global_null[metric].to_numpy(dtype=int)
bins = np.arange(values.min()-0.5, values.max()+1.5, 1)
ax.hist(values, bins=bins, alpha=0.8)
ax.axvline(lgg_global_observed_metrics[metric], linewidth=2.5, color=DISSERTATION_NAVY,
           label=f"Observed = {lgg_global_observed_metrics[metric]}")
ax.set_xlabel("Large (≥4-gene) components represented by ≥4 panels")
ax.set_ylabel("Five-panel null replicate count")
ax.set_title("LGG Global Five-Panel Null: Multi-Panel Component Structure")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "LGG_global_null_large_components_fourplus_panels", NULL_DIR)
plt.show()
plt.close(fig)


## 12. LGG cross-seed correlation network and correlation-component stability

Edges are created only when a pair of distinct genes occurs across at least one pair of different evolutionary seeds and has training $|r|$ at or above the primary threshold. Connected components summarize recurrent correlated signal structure. Components are **not automatically assigned biological pathway labels**; a template is written for later annotation using the pathway results.

In [ ]:
def build_cross_seed_edge_table(panels, corr_matrix, gene_to_index, threshold):
    """
    Build the cross-seed substitution network. A non-identical gene pair is eligible
    only when the genes have disjoint seed membership, so the edge cannot be explained
    by the two genes being co-selected in any one final panel. This reproduces the
    dissertation cross-seed network definition: four major components plus the small
    two-gene components are based on between-panel rather than within-panel correlation.
    """
    all_genes = sorted(set().union(*[set(v) for v in panels.values()]))
    membership = {g: {s for s, panel in panels.items() if g in panel} for g in all_genes}
    rows = []
    for i, ga in enumerate(all_genes):
        for gb in all_genes[i+1:]:
            if not membership[ga].isdisjoint(membership[gb]):
                continue
            r = float(corr_matrix[gene_to_index[ga], gene_to_index[gb]])
            if abs(r) < threshold:
                continue
            seed_pairs = sorted((sa, sb) for sa in membership[ga] for sb in membership[gb])
            rows.append({
                "gene_a": ga,
                "gene_b": gb,
                "r_training": r,
                "abs_r_training": abs(r),
                "supporting_seed_pairs": ";".join(f"{a}-{b}" for a, b in seed_pairs),
                "number_of_supporting_seed_pairs": len(seed_pairs),
            })
    return pd.DataFrame(rows).sort_values("abs_r_training", ascending=False, kind="stable") if rows else pd.DataFrame()

lgg_network_edges = build_cross_seed_edge_table(
    lgg_panels, lgg_train_corr, lgg_gene_to_index, PRIMARY_CORRELATION_THRESHOLD
)
lgg_network_edges["gene_name_a"] = lgg_network_edges["gene_a"].map(lgg_gene_to_symbol)
lgg_network_edges["gene_name_b"] = lgg_network_edges["gene_b"].map(lgg_gene_to_symbol)
lgg_network_edges.to_csv(PROGRAM_DIR / "LGG_cross_seed_correlation_edges.csv", index=False)

all_selected_lgg = sorted(set().union(*map(set, lgg_panels.values())))
gene_seed_membership = {g: sorted([s for s, panel in lgg_panels.items() if g in panel]) for g in all_selected_lgg}

G = nx.Graph()
for g in all_selected_lgg:
    G.add_node(g)
for _, row in lgg_network_edges.iterrows():
    G.add_edge(row["gene_a"], row["gene_b"], weight=row["abs_r_training"], r=row["r_training"])

components = sorted(nx.connected_components(G), key=lambda c: (-len(c), sorted(c)[0]))
component_id = {}
for idx, comp in enumerate(components, start=1):
    for gene in comp:
        component_id[gene] = idx

node_rows = []
for g in all_selected_lgg:
    node_rows.append({
        "gene_id": g,
        "gene_name": lgg_gene_to_symbol.get(g, ""),
        "selected_in_seeds": ";".join(map(str, gene_seed_membership[g])),
        "seed_count": len(gene_seed_membership[g]),
        "degree": G.degree(g),
        "component_id": component_id[g],
        "component_size": len(components[component_id[g]-1]),
    })
lgg_network_nodes = pd.DataFrame(node_rows)
lgg_network_nodes.to_csv(PROGRAM_DIR / "LGG_cross_seed_correlation_nodes.csv", index=False)

component_rows = []
for idx, comp in enumerate(components, start=1):
    sub = G.subgraph(comp)
    represented = sorted({s for g in comp for s in gene_seed_membership[g]})
    component_rows.append({
        "component_id": idx,
        "genes": len(comp),
        "edges": sub.number_of_edges(),
        "seeds_represented": len(represented),
        "seed_ids": ";".join(map(str, represented)),
        "gene_symbols": ";".join(sorted(gene_symbol(g, lgg_gene_to_symbol) for g in comp)),
        "large_component_ge_4_genes": len(comp) >= NETWORK_MIN_COMPONENT_SIZE,
    })
lgg_components = pd.DataFrame(component_rows)
lgg_components.to_csv(PROGRAM_DIR / "LGG_correlation_component_summary.csv", index=False)

# Seed-by-component representation matrix: did a seed select at least one gene in the component?
large_components = lgg_components.loc[lgg_components["large_component_ge_4_genes"], "component_id"].tolist()
representation_rows = []
for cid in large_components:
    comp_genes = components[cid-1]
    for seed in LGG_SEEDS:
        selected = sorted(set(lgg_panels[seed]) & set(comp_genes))
        representation_rows.append({
            "component_id": cid,
            "seed": seed,
            "represented": len(selected) > 0,
            "selected_gene_count_from_component": len(selected),
            "selected_genes": ";".join(gene_symbol(g, lgg_gene_to_symbol) for g in selected),
        })
lgg_component_representation = pd.DataFrame(representation_rows)
lgg_component_representation.to_csv(PROGRAM_DIR / "LGG_large_component_representation_by_seed.csv", index=False)

annotation_template = lgg_components.loc[lgg_components["large_component_ge_4_genes"], [
    "component_id", "genes", "edges", "seeds_represented", "gene_symbols"
]].copy()
annotation_template["biological_program_label"] = ""
annotation_template["functional_notes"] = ""
annotation_template.to_csv(PROGRAM_DIR / "LGG_component_annotation_template.csv", index=False)

display(lgg_components.head(12))
print(f"Cross-seed network: {G.number_of_nodes()} selected genes, {G.number_of_edges()} edges")
print(f"Components with >= {NETWORK_MIN_COMPONENT_SIZE} genes: {len(large_components)}")

## 13. Bootstrap robustness of the training-defined LGG correlation edges

Only the cross-seed edges that passed the training threshold are bootstrapped. Patient rows are sampled with replacement from the 197-patient training cohort. The output reports the fraction of bootstrap samples in which each edge remains above the primary $|r|$ threshold.

In [ ]:
def _bootstrap_corr_chunk(X, idx_a, idx_b, n_rep, seed):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    E = len(idx_a)
    out = np.empty((n_rep, E), dtype=np.float32)
    for b in range(n_rep):
        rows = rng.integers(0, n, size=n)
        A = X[rows][:, idx_a].T.astype(np.float64, copy=False)
        B = X[rows][:, idx_b].T.astype(np.float64, copy=False)
        A -= A.mean(axis=1, keepdims=True)
        B -= B.mean(axis=1, keepdims=True)
        denom = np.sqrt(np.sum(A*A, axis=1) * np.sum(B*B, axis=1))
        corr = np.divide(np.sum(A*B, axis=1), denom, out=np.full(E, np.nan), where=denom>0)
        out[b] = corr.astype(np.float32)
    return out

if lgg_network_edges.empty:
    print("No network edges available for bootstrap analysis.")
    lgg_bootstrap_edge_summary = pd.DataFrame()
else:
    bootstrap_cache = BOOTSTRAP_DIR / f"LGG_edge_bootstrap_summary_B{N_CORRELATION_BOOTSTRAPS}.csv"
    if bootstrap_cache.exists():
        lgg_bootstrap_edge_summary = pd.read_csv(bootstrap_cache)
        print("Loaded cached bootstrap summary.")
    else:
        X_lgg = lgg_train[lgg_candidate_genes].to_numpy(dtype=np.float32)
        idx_a = np.array([lgg_gene_to_index[g] for g in lgg_network_edges["gene_a"]], dtype=int)
        idx_b = np.array([lgg_gene_to_index[g] for g in lgg_network_edges["gene_b"]], dtype=int)
        counts = np.full(N_WORKERS, N_CORRELATION_BOOTSTRAPS // N_WORKERS, dtype=int)
        counts[: N_CORRELATION_BOOTSTRAPS % N_WORKERS] += 1
        jobs = [(int(n), RANDOM_SEED + 2000 + j) for j, n in enumerate(counts) if n > 0]
        print(f"Bootstrapping {len(idx_a):,} LGG edges across {N_CORRELATION_BOOTSTRAPS:,} patient resamples...")
        parts = Parallel(
            n_jobs=N_WORKERS, backend=JOBLIB_BACKEND,
            temp_folder=str(JOBLIB_TEMP_DIR), max_nbytes="10M", mmap_mode="r"
        )(
            delayed(_bootstrap_corr_chunk)(X_lgg, idx_a, idx_b, n, seed)
            for n, seed in jobs
        )
        boot = np.vstack(parts)
        q025, q50, q975 = np.nanquantile(boot, [0.025, 0.50, 0.975], axis=0)
        abs_boot = np.abs(boot)
        lgg_bootstrap_edge_summary = lgg_network_edges.copy().reset_index(drop=True)
        lgg_bootstrap_edge_summary["bootstrap_median_r"] = q50
        lgg_bootstrap_edge_summary["bootstrap_2.5pct_r"] = q025
        lgg_bootstrap_edge_summary["bootstrap_97.5pct_r"] = q975
        lgg_bootstrap_edge_summary["bootstrap_fraction_abs_r_ge_threshold"] = (
            abs_boot >= PRIMARY_CORRELATION_THRESHOLD
        ).mean(axis=0)
        lgg_bootstrap_edge_summary["bootstrap_sign_agreement_fraction"] = (
            np.sign(boot) == np.sign(lgg_network_edges["r_training"].to_numpy())[None, :]
        ).mean(axis=0)
        lgg_bootstrap_edge_summary.to_csv(bootstrap_cache, index=False)

    display(lgg_bootstrap_edge_summary.head(10))
    fig, ax = plt.subplots(figsize=(7.0, 4.8))
    ax.hist(lgg_bootstrap_edge_summary["bootstrap_fraction_abs_r_ge_threshold"], bins=20)
    ax.set_xlabel(f"Bootstrap fraction with |r| ≥ {PRIMARY_CORRELATION_THRESHOLD:.2f}")
    ax.set_ylabel("Training-defined cross-seed edges")
    ax.set_title("LGG Bootstrap Robustness of Cross-Seed Correlation Edges")
    fig.tight_layout()
    save_figure(fig, "LGG_bootstrap_edge_threshold_retention", BOOTSTRAP_DIR)
    plt.show()
    plt.close(fig)

## 14. Held-out preservation of the entire training-defined LGG correlation network

This evaluates all training-defined network edges in the 85 held-out patients, including edges that were not selected by the one-to-one matching algorithm.

In [ ]:
if not lgg_network_edges.empty:
    network_pairs = list(zip(lgg_network_edges["gene_a"], lgg_network_edges["gene_b"]))
    network_valid_r = direct_pair_correlations(lgg_valid.set_index("case_submitter_id"), network_pairs)
    network_preservation = lgg_network_edges.copy()
    network_preservation["r_heldout_validation"] = network_valid_r
    network_preservation["abs_r_heldout_validation"] = np.abs(network_valid_r)
    network_preservation["preserved_at_same_threshold"] = (
        network_preservation["abs_r_heldout_validation"] >= PRIMARY_CORRELATION_THRESHOLD
    )
    network_preservation["component_id"] = network_preservation["gene_a"].map(component_id)
    network_preservation.to_csv(PRESERVATION_DIR / "LGG_network_edge_heldout_preservation.csv", index=False)

    component_preservation = network_preservation.groupby("component_id").agg(
        training_edges=("gene_a", "size"),
        preserved_edges=("preserved_at_same_threshold", "sum"),
        preservation_fraction=("preserved_at_same_threshold", "mean"),
        median_abs_r_training=("abs_r_training", "median"),
        median_abs_r_validation=("abs_r_heldout_validation", "median"),
    ).reset_index()
    component_preservation.to_csv(PRESERVATION_DIR / "LGG_component_edge_heldout_preservation.csv", index=False)
    display(component_preservation.head(12))

## 15. Secondary predictive reproducibility of the five LGG selected panels

For each already selected 50-gene panel, the fixed histogram gradient boosted classifier used for NSGA-II fitness is fit once on all 197 training patients and evaluated on the 85 held-out patients. **No panel is selected or discarded on the basis of these results.** This is a secondary post hoc reproducibility analysis for Paper 2.

In [ ]:
LGG_CLASSES = ["IDHmut-codel", "IDHmut-non-codel", "IDHwt"]
FITNESS_MODEL_PARAMETERS = {
    "learning_rate": 0.01,
    "max_iter": 50,
    "max_depth": 5,
    "min_samples_leaf": 10,
    "max_bins": 20,
    "l2_regularization": 0.0,
    "early_stopping": False,
    "random_state": 42,
}

split_labels = lgg_split.set_index("case_submitter_id")["molecular_subtype"]
encoder = LabelEncoder().fit(LGG_CLASSES)
y_train_lgg = encoder.transform(split_labels.loc[lgg_train["case_submitter_id"]])
y_valid_lgg = encoder.transform(split_labels.loc[lgg_valid["case_submitter_id"]])

predictive_rows = []
for seed in LGG_SEEDS:
    genes = lgg_panels[seed]
    Xtr = lgg_train[genes].to_numpy(dtype=np.float32)
    Xva = lgg_valid[genes].to_numpy(dtype=np.float32)
    model = HistGradientBoostingClassifier(**FITNESS_MODEL_PARAMETERS)
    weights = compute_sample_weight(class_weight="balanced", y=y_train_lgg)
    with threadpool_limits(limits=1):
        model.fit(Xtr, y_train_lgg, sample_weight=weights)
        pred = model.predict(Xva)
        proba = model.predict_proba(Xva)
    row_summary = lgg_selected_summary.loc[lgg_selected_summary["evolutionary_seed"] == seed].iloc[0]
    predictive_rows.append({
        "evolutionary_seed": seed,
        "selected_training_CV_balanced_accuracy_from_original_run": row_summary["cv_balanced_accuracy"],
        "heldout_accuracy_fixed_HGB": accuracy_score(y_valid_lgg, pred),
        "heldout_balanced_accuracy_fixed_HGB": balanced_accuracy_score(y_valid_lgg, pred),
        "heldout_macro_precision_fixed_HGB": precision_score(y_valid_lgg, pred, average="macro", zero_division=0),
        "heldout_macro_recall_fixed_HGB": recall_score(y_valid_lgg, pred, average="macro", zero_division=0),
        "heldout_macro_f1_fixed_HGB": f1_score(y_valid_lgg, pred, average="macro", zero_division=0),
        "heldout_macro_ovr_roc_auc_fixed_HGB": roc_auc_score(y_valid_lgg, proba, multi_class="ovr", average="macro"),
    })

lgg_predictive_reproducibility = pd.DataFrame(predictive_rows)
lgg_predictive_reproducibility.to_csv(PREDICTIVE_DIR / "LGG_five_seed_panel_fixed_HGB_heldout_performance.csv", index=False)

display(lgg_predictive_reproducibility)

fig, ax = plt.subplots(figsize=(7.0, 4.8))
ax.plot(lgg_predictive_reproducibility["evolutionary_seed"].astype(str),
        lgg_predictive_reproducibility["heldout_balanced_accuracy_fixed_HGB"], marker="o")
ax.set_xlabel("Evolutionary seed")
ax.set_ylabel("Held-out balanced accuracy")
ax.set_ylim(0, 1.02)
ax.set_title("LGG Predictive Reproducibility of Five Selected Panels")
fig.tight_layout()
save_figure(fig, "LGG_five_seed_fixed_HGB_heldout_balanced_accuracy", PREDICTIVE_DIR)
plt.show()
plt.close(fig)

# Part II — Kidney TPM-versus-CPM analytical representation perturbation

The two final kidney panels were selected from different training-only 3,000-gene candidate pools. Therefore, Nogueira stability is **not** treated as directly comparable to the LGG multiseed estimate. The kidney analysis instead emphasizes exact overlap, candidate-pool overlap, correlation-adjusted one-to-one matching, held-out preservation, and the already completed held-out predictive results.

## 16. Kidney exact selected-panel and candidate-pool overlap

In [ ]:
kidney_candidate_overlap = len(set(kidney_tpm_candidates) & set(kidney_cpm_candidates))
kidney_candidate_union = len(set(kidney_tpm_candidates) | set(kidney_cpm_candidates))
kidney_panel_overlap = len(set(kidney_tpm_panel) & set(kidney_cpm_panel))
kidney_panel_union = len(set(kidney_tpm_panel) | set(kidney_cpm_panel))

kidney_exact = pd.DataFrame([{
    "TPM_candidate_count": len(kidney_tpm_candidates),
    "CPM_candidate_count": len(kidney_cpm_candidates),
    "candidate_overlap_count": kidney_candidate_overlap,
    "candidate_Jaccard": kidney_candidate_overlap / kidney_candidate_union,
    "TPM_panel_count": len(kidney_tpm_panel),
    "CPM_panel_count": len(kidney_cpm_panel),
    "selected_panel_overlap_count": kidney_panel_overlap,
    "selected_panel_Jaccard": kidney_panel_overlap / kidney_panel_union,
}])
kidney_exact.to_csv(KIDNEY_DIR / "Kidney_TPM_CPM_exact_overlap_summary.csv", index=False)

kidney_performance.to_csv(KIDNEY_DIR / "Kidney_TPM_CPM_existing_predictive_performance.csv", index=False)
display(kidney_exact)
display(kidney_performance)

## 17. Load only the kidney genes needed for correlation analyses

The full source TPM and CPM matrices are read using `usecols`, so only the union of the two 3,000-gene candidate pools is retained in memory. Values are transformed to log2(value + 1), matching the two modeling branches.

In [ ]:
kidney_candidate_union_genes = sorted(set(kidney_tpm_candidates) | set(kidney_cpm_candidates))


def read_kidney_full_subset(path, genes):
    usecols = ["case_submitter_id", *genes]
    df = pd.read_csv(path, usecols=usecols)
    df = df.set_index("case_submitter_id")
    # Source files contain TPM or CPM before log transformation.
    df.loc[:, genes] = np.log2(df.loc[:, genes].astype(np.float32) + np.float32(1.0))
    return df

print(f"Kidney candidate-pool union contains {len(kidney_candidate_union_genes):,} genes.")
print("Loading full-source TPM subset...")
kidney_full_log_tpm = read_kidney_full_subset(
    KIDNEY_PROCESSED_DIR / "tcga_kidney_tpm.csv.gz", kidney_candidate_union_genes
)
print("Loading full-source CPM subset...")
kidney_full_log_cpm = read_kidney_full_subset(
    KIDNEY_PROCESSED_DIR / "tcga_kidney_cpm.csv.gz", kidney_candidate_union_genes
)

# Preserve the exact split order from metadata.
kidney_tpm_train_union = kidney_full_log_tpm.loc[kidney_train_ids, kidney_candidate_union_genes]
kidney_tpm_valid_union = kidney_full_log_tpm.loc[kidney_valid_ids, kidney_candidate_union_genes]
kidney_cpm_train_union = kidney_full_log_cpm.loc[kidney_train_ids, kidney_candidate_union_genes]
kidney_cpm_valid_union = kidney_full_log_cpm.loc[kidney_valid_ids, kidney_candidate_union_genes]

print("Loaded kidney correlation inputs:")
print("TPM training:", kidney_tpm_train_union.shape)
print("TPM validation:", kidney_tpm_valid_union.shape)
print("CPM training:", kidney_cpm_train_union.shape)
print("CPM validation:", kidney_cpm_valid_union.shape)

## 18. Kidney candidate-pool cross-correlation matrices

Rows correspond to the TPM branch's 3,000 candidate genes and columns correspond to the CPM branch's 3,000 candidate genes. Three training-only matching spaces are analyzed:

1. correlations measured in log2(TPM+1);
2. correlations measured in log2(CPM+1);
3. a conservative **consensus** score equal to the minimum of the two absolute correlations, so an edge qualifies only if it is strong under both representations.

In [ ]:
KIDNEY_TPM_CROSS_CACHE = CACHE_DIR / "Kidney_TPMcandidate_vs_CPMcandidate_corr_in_logTPM_float32.npy"
KIDNEY_CPM_CROSS_CACHE = CACHE_DIR / "Kidney_TPMcandidate_vs_CPMcandidate_corr_in_logCPM_float32.npy"

if KIDNEY_TPM_CROSS_CACHE.exists():
    kidney_cross_tpm = np.load(KIDNEY_TPM_CROSS_CACHE, mmap_mode="r")
else:
    print("Computing kidney 3,000 x 3,000 cross-correlation matrix in TPM representation...")
    Xa = kidney_tpm_train_union[kidney_tpm_candidates].to_numpy(dtype=np.float32)
    Xb = kidney_tpm_train_union[kidney_cpm_candidates].to_numpy(dtype=np.float32)
    with threadpool_limits(limits=max(1, min(N_WORKERS, 10))):
        corr = cross_correlation_matrix(Xa, Xb)
    np.save(KIDNEY_TPM_CROSS_CACHE, corr.astype(np.float32))
    kidney_cross_tpm = np.load(KIDNEY_TPM_CROSS_CACHE, mmap_mode="r")

if KIDNEY_CPM_CROSS_CACHE.exists():
    kidney_cross_cpm = np.load(KIDNEY_CPM_CROSS_CACHE, mmap_mode="r")
else:
    print("Computing kidney 3,000 x 3,000 cross-correlation matrix in CPM representation...")
    Xa = kidney_cpm_train_union[kidney_tpm_candidates].to_numpy(dtype=np.float32)
    Xb = kidney_cpm_train_union[kidney_cpm_candidates].to_numpy(dtype=np.float32)
    with threadpool_limits(limits=max(1, min(N_WORKERS, 10))):
        corr = cross_correlation_matrix(Xa, Xb)
    np.save(KIDNEY_CPM_CROSS_CACHE, corr.astype(np.float32))
    kidney_cross_cpm = np.load(KIDNEY_CPM_CROSS_CACHE, mmap_mode="r")

kidney_abs_cross_tpm = np.abs(kidney_cross_tpm)
kidney_abs_cross_cpm = np.abs(kidney_cross_cpm)
# This array is only ~36 MB at float32 and is intentionally not written as a third cache.
kidney_abs_cross_consensus = np.minimum(kidney_abs_cross_tpm, kidney_abs_cross_cpm)

tpm_candidate_index = {g: i for i, g in enumerate(kidney_tpm_candidates)}
cpm_candidate_index = {g: i for i, g in enumerate(kidney_cpm_candidates)}

print("Kidney cross-correlation matrices ready.")

## 19. Kidney correlation-adjusted TPM-versus-CPM panel similarity across thresholds

In [ ]:
kidney_matching_rows = []
kidney_primary_match_tables = {}

for representation, abs_matrix in [
    ("log2TPM1", kidney_abs_cross_tpm),
    ("log2CPM1", kidney_abs_cross_cpm),
    ("consensus_min_abs_r", kidney_abs_cross_consensus),
]:
    for tau in CORRELATION_THRESHOLDS:
        summary, pairs = panel_match(
            kidney_tpm_panel, kidney_cpm_panel,
            tpm_candidate_index, cpm_candidate_index,
            abs_matrix, tau
        )
        kidney_matching_rows.append({
            "representation": representation,
            "threshold": tau,
            "exact_selected_panel_jaccard": jaccard(kidney_tpm_panel, kidney_cpm_panel),
            **summary,
        })
        if np.isclose(tau, PRIMARY_CORRELATION_THRESHOLD):
            pairs = pairs.copy()
            if not pairs.empty:
                pairs["TPM_gene_name"] = pairs["gene_a"].map(kidney_gene_to_symbol)
                pairs["CPM_gene_name"] = pairs["gene_b"].map(kidney_gene_to_symbol)
            kidney_primary_match_tables[representation] = pairs
            pairs.to_csv(
                KIDNEY_DIR / f"Kidney_TPM_CPM_matches_{representation}_tau_{tau:.2f}.csv",
                index=False,
            )

kidney_corr_adjusted = pd.DataFrame(kidney_matching_rows)
kidney_corr_adjusted.to_csv(KIDNEY_DIR / "Kidney_TPM_CPM_correlation_adjusted_similarity_by_threshold.csv", index=False)
display(kidney_corr_adjusted.loc[np.isclose(kidney_corr_adjusted["threshold"], PRIMARY_CORRELATION_THRESHOLD)])

fig, ax = plt.subplots(figsize=(7.4, 5.1))
for representation, group in kidney_corr_adjusted.groupby("representation"):
    ax.plot(group["threshold"], group["correlation_adjusted_jaccard"], marker="o", label=representation)
ax.axvline(PRIMARY_CORRELATION_THRESHOLD, linestyle="--", linewidth=1, color="black")
ax.set_xlabel("Absolute correlation threshold")
ax.set_ylabel("Correlation-adjusted Jaccard")
ax.set_ylim(0, 1.02)
ax.set_title("Kidney TPM-versus-CPM Correlation-Adjusted Panel Similarity")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "Kidney_TPM_CPM_corr_adjusted_Jaccard_threshold_curve", KIDNEY_DIR)
plt.show()
plt.close(fig)

## 20. Kidney training-to-held-out and cross-representation preservation

For each training-defined one-to-one match, this cell reports the correlation for the same gene pair in both TPM and CPM representations and in both training and held-out patients. This allows a pair found under one representation to be checked under the other representation without rematching.

In [ ]:
def add_kidney_pair_correlations(match_df, source_label):
    if match_df.empty:
        return pd.DataFrame()
    pairs = list(zip(match_df["gene_a"], match_df["gene_b"]))
    out = match_df.copy()
    out.insert(0, "matching_source", source_label)
    out["r_TPM_training"] = direct_pair_correlations(kidney_tpm_train_union, pairs)
    out["r_TPM_validation"] = direct_pair_correlations(kidney_tpm_valid_union, pairs)
    out["r_CPM_training"] = direct_pair_correlations(kidney_cpm_train_union, pairs)
    out["r_CPM_validation"] = direct_pair_correlations(kidney_cpm_valid_union, pairs)
    for col in ["r_TPM_training", "r_TPM_validation", "r_CPM_training", "r_CPM_validation"]:
        out["abs_" + col] = np.abs(out[col])
    out["preserved_TPM_validation"] = out["abs_r_TPM_validation"] >= PRIMARY_CORRELATION_THRESHOLD
    out["preserved_CPM_validation"] = out["abs_r_CPM_validation"] >= PRIMARY_CORRELATION_THRESHOLD
    out["strong_in_both_training_representations"] = (
        (out["abs_r_TPM_training"] >= PRIMARY_CORRELATION_THRESHOLD) &
        (out["abs_r_CPM_training"] >= PRIMARY_CORRELATION_THRESHOLD)
    )
    out["strong_in_both_validation_representations"] = (
        (out["abs_r_TPM_validation"] >= PRIMARY_CORRELATION_THRESHOLD) &
        (out["abs_r_CPM_validation"] >= PRIMARY_CORRELATION_THRESHOLD)
    )
    return out

kidney_preservation_parts = []
for source, df in kidney_primary_match_tables.items():
    part = add_kidney_pair_correlations(df, source)
    if not part.empty:
        kidney_preservation_parts.append(part)

kidney_pair_preservation = pd.concat(kidney_preservation_parts, ignore_index=True) if kidney_preservation_parts else pd.DataFrame()
kidney_pair_preservation.to_csv(KIDNEY_DIR / "Kidney_training_matches_cross_representation_and_heldout_preservation.csv", index=False)

if not kidney_pair_preservation.empty:
    kidney_preservation_summary = kidney_pair_preservation.groupby("matching_source").agg(
        training_matches=("gene_a", "size"),
        fraction_strong_in_both_training_representations=("strong_in_both_training_representations", "mean"),
        fraction_preserved_TPM_validation=("preserved_TPM_validation", "mean"),
        fraction_preserved_CPM_validation=("preserved_CPM_validation", "mean"),
        fraction_strong_in_both_validation_representations=("strong_in_both_validation_representations", "mean"),
    ).reset_index()
    kidney_preservation_summary.to_csv(KIDNEY_DIR / "Kidney_match_preservation_summary.csv", index=False)
    display(kidney_preservation_summary)

## 21. Kidney null model matched to the two branch-specific candidate pools

Each null replicate draws 50 genes from the TPM candidate pool and 50 genes from the CPM candidate pool. The same random draw is evaluated in TPM, CPM, and consensus correlation spaces. This preserves the actual overlap structure of the two 3,000-gene candidate universes.

In [ ]:
def _null_kidney_chunk(n_rep, seed, abs_tpm, abs_cpm, tpm_genes, cpm_genes, threshold, k=50):
    rng = np.random.default_rng(seed)
    out = np.empty((n_rep, 8), dtype=float)
    for i in range(n_rep):
        ia = rng.choice(len(tpm_genes), size=k, replace=False)
        ib = rng.choice(len(cpm_genes), size=k, replace=False)
        genes_a = {tpm_genes[j] for j in ia}
        genes_b = {cpm_genes[j] for j in ib}
        exact = len(genes_a & genes_b)
        M_tpm = np.asarray(abs_tpm[np.ix_(ia, ib)], dtype=float)
        M_cpm = np.asarray(abs_cpm[np.ix_(ia, ib)], dtype=float)
        M_con = np.minimum(M_tpm, M_cpm)
        s_tpm = matching_summary(M_tpm, threshold, k, k)
        s_cpm = matching_summary(M_cpm, threshold, k, k)
        s_con = matching_summary(M_con, threshold, k, k)
        out[i] = [
            exact,
            exact / (2*k-exact),
            s_tpm["matched_count"], s_tpm["correlation_adjusted_jaccard"],
            s_cpm["matched_count"], s_cpm["correlation_adjusted_jaccard"],
            s_con["matched_count"], s_con["correlation_adjusted_jaccard"],
        ]
    return out


def simulate_kidney_null(n_replicates, threshold):
    cache = KIDNEY_DIR / f"Kidney_null_n{n_replicates}_tau{threshold:.2f}.csv"
    if cache.exists():
        df = pd.read_csv(cache)
        if len(df) == n_replicates:
            print("Loading cached kidney null distribution:", cache.name)
            return df
    counts = np.full(N_WORKERS, n_replicates // N_WORKERS, dtype=int)
    counts[: n_replicates % N_WORKERS] += 1
    jobs = [(int(n), RANDOM_SEED + 3000 + j) for j, n in enumerate(counts) if n > 0]
    print(f"Simulating {n_replicates:,} kidney null panel pairs using {len(jobs)} workers...")
    parts = Parallel(
        n_jobs=N_WORKERS, backend=JOBLIB_BACKEND,
        temp_folder=str(JOBLIB_TEMP_DIR), max_nbytes="10M", mmap_mode="r"
    )(
        delayed(_null_kidney_chunk)(
            n, seed, kidney_abs_cross_tpm, kidney_abs_cross_cpm,
            kidney_tpm_candidates, kidney_cpm_candidates,
            threshold
        ) for n, seed in jobs
    )
    arr = np.vstack(parts)
    df = pd.DataFrame(arr, columns=[
        "exact_overlap", "exact_jaccard",
        "TPM_matched_count", "TPM_corr_adjusted_jaccard",
        "CPM_matched_count", "CPM_corr_adjusted_jaccard",
        "consensus_matched_count", "consensus_corr_adjusted_jaccard",
    ])
    df.insert(0, "null_replicate", np.arange(1, len(df)+1))
    df.to_csv(cache, index=False)
    return df

kidney_null = simulate_kidney_null(N_NULL_REPLICATES, PRIMARY_CORRELATION_THRESHOLD)

kidney_observed_primary = kidney_corr_adjusted.loc[
    np.isclose(kidney_corr_adjusted["threshold"], PRIMARY_CORRELATION_THRESHOLD)
].copy()

kidney_null_test_rows = []
column_map = {
    "log2TPM1": "TPM_corr_adjusted_jaccard",
    "log2CPM1": "CPM_corr_adjusted_jaccard",
    "consensus_min_abs_r": "consensus_corr_adjusted_jaccard",
}
for _, row in kidney_observed_primary.iterrows():
    null_col = column_map[row["representation"]]
    p = (1 + np.sum(kidney_null[null_col] >= row["correlation_adjusted_jaccard"])) / (len(kidney_null) + 1)
    kidney_null_test_rows.append({
        "representation": row["representation"],
        "observed_correlation_adjusted_jaccard": row["correlation_adjusted_jaccard"],
        "empirical_p_null_ge_observed": p,
        "null_replicates": len(kidney_null),
    })
kidney_null_tests = pd.DataFrame(kidney_null_test_rows)
kidney_null_tests.to_csv(KIDNEY_DIR / "Kidney_observed_vs_random_panel_null_tests.csv", index=False)
display(kidney_null_tests)

## 21A. Kidney chance-corrected correlation-adjusted stability across thresholds

Because the TPM and CPM branches have different 3,000-gene candidate pools, the kidney null draws one 50-gene panel from each branch-specific candidate universe. Null expectations are calculated separately for correlations measured in log2(TPM+1), log2(CPM+1), and the conservative consensus defined by the minimum absolute correlation across both representations. Chance correction is then applied at every threshold.


In [ ]:

def _null_kidney_threshold_curve_chunk(n_rep, seed, abs_tpm, abs_cpm, thresholds, k=50):
    rng = np.random.default_rng(seed)
    thresholds = np.asarray(thresholds, dtype=float)
    out = np.empty((n_rep, len(thresholds) * 3), dtype=np.float32)
    for i in range(n_rep):
        ia = rng.choice(abs_tpm.shape[0], size=k, replace=False)
        ib = rng.choice(abs_tpm.shape[1], size=k, replace=False)
        M_tpm = np.asarray(abs_tpm[np.ix_(ia, ib)], dtype=float)
        M_cpm = np.asarray(abs_cpm[np.ix_(ia, ib)], dtype=float)
        M_con = np.minimum(M_tpm, M_cpm)
        for j, tau in enumerate(thresholds):
            for rep_i, M in enumerate((M_tpm, M_cpm, M_con)):
                m = maximum_cardinality_match_count(M, tau)
                out[i, rep_i * len(thresholds) + j] = corr_adjusted_jaccard_from_match_count(m, k, k)
    return out


def simulate_kidney_null_threshold_curve(n_replicates, thresholds):
    thresholds = [float(t) for t in thresholds]
    cache = KIDNEY_DIR / f"Kidney_pairwise_null_threshold_curve_n{n_replicates}.csv"
    labels = ["log2TPM1", "log2CPM1", "consensus_min_abs_r"]
    expected_cols = [f"{label}_tau_{tau:.2f}" for label in labels for tau in thresholds]
    if cache.exists():
        existing = pd.read_csv(cache)
        if len(existing) == n_replicates and all(c in existing.columns for c in expected_cols):
            print("Loading cached kidney null threshold curve:", cache.name)
            return existing
    counts = np.full(N_WORKERS, n_replicates // N_WORKERS, dtype=int)
    counts[: n_replicates % N_WORKERS] += 1
    jobs = [(int(n), RANDOM_SEED + 4000 + j) for j, n in enumerate(counts) if n > 0]
    print(f"Simulating kidney null similarity across {len(thresholds)} thresholds and {n_replicates:,} panel pairs...")
    parts = Parallel(
        n_jobs=N_WORKERS, backend=JOBLIB_BACKEND,
        temp_folder=str(JOBLIB_TEMP_DIR), max_nbytes="10M", mmap_mode="r"
    )(
        delayed(_null_kidney_threshold_curve_chunk)(
            n, seed, kidney_abs_cross_tpm, kidney_abs_cross_cpm, thresholds
        ) for n, seed in jobs
    )
    arr = np.vstack(parts)
    df = pd.DataFrame(arr, columns=expected_cols)
    df.insert(0, "null_replicate", np.arange(1, len(df)+1))
    df.to_csv(cache, index=False)
    return df


kidney_null_curve = simulate_kidney_null_threshold_curve(N_NULL_REPLICATES, CORRELATION_THRESHOLDS)

kidney_null_curve_summary_rows = []
for representation in ["log2TPM1", "log2CPM1", "consensus_min_abs_r"]:
    for tau in CORRELATION_THRESHOLDS:
        values = kidney_null_curve[f"{representation}_tau_{tau:.2f}"].to_numpy(dtype=float)
        kidney_null_curve_summary_rows.append({
            "representation": representation,
            "threshold": tau,
            "null_mean_corr_adjusted_jaccard": float(np.mean(values)),
            "null_median_corr_adjusted_jaccard": float(np.median(values)),
            "null_q025_corr_adjusted_jaccard": float(np.quantile(values, 0.025)),
            "null_q975_corr_adjusted_jaccard": float(np.quantile(values, 0.975)),
            "null_replicates": len(values),
        })
kidney_null_curve_summary = pd.DataFrame(kidney_null_curve_summary_rows)
kidney_null_curve_summary.to_csv(
    KIDNEY_DIR / "Kidney_pairwise_null_threshold_curve_summary.csv", index=False
)

kidney_corr_adjusted_chance = kidney_corr_adjusted.merge(
    kidney_null_curve_summary[["representation", "threshold", "null_mean_corr_adjusted_jaccard"]],
    on=["representation", "threshold"], how="left"
)
kidney_corr_adjusted_chance["chance_corrected_corr_adjusted_stability"] = chance_corrected_similarity(
    kidney_corr_adjusted_chance["correlation_adjusted_jaccard"],
    kidney_corr_adjusted_chance["null_mean_corr_adjusted_jaccard"],
)
kidney_corr_adjusted_chance.to_csv(
    KIDNEY_DIR / "Kidney_TPM_CPM_chance_corrected_correlation_adjusted_similarity_by_threshold.csv",
    index=False,
)
display(kidney_corr_adjusted_chance.loc[
    np.isclose(kidney_corr_adjusted_chance["threshold"], PRIMARY_CORRELATION_THRESHOLD)
])

fig, ax = plt.subplots(figsize=(7.5, 5.2))
for representation, group in kidney_corr_adjusted_chance.groupby("representation"):
    ax.plot(group["threshold"], group["chance_corrected_corr_adjusted_stability"],
            marker="o", linewidth=2, label=representation)
ax.axhline(0.0, linestyle="--", linewidth=1, color="black")
ax.axvline(PRIMARY_CORRELATION_THRESHOLD, linestyle="--", linewidth=1, color="black")
ax.set_xlabel("Absolute correlation threshold")
ax.set_ylabel("Chance-corrected correlation-adjusted stability")
ax.set_title("Kidney TPM-versus-CPM Chance-Corrected Stability")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "Kidney_TPM_CPM_chance_corrected_corr_adjusted_threshold_curve", KIDNEY_DIR)
plt.show()
plt.close(fig)


## 22. Kidney fixed-model predictive stability comparison

The original kidney manuscript analysis used a post-selection CART decision tree, but that creates a model-family difference relative to the LGG five-panel predictive reproducibility analysis. This revision therefore performs an additional **fixed histogram gradient boosted classifier** comparison using the same classifier parameters used for NSGA-II fitness. Each final 50-gene panel is refit once on all 622 training patients in its own expression representation and evaluated on the untouched 267-patient validation cohort.

The original CART metrics are retained as secondary context, but the fixed-HGB comparison is the directly analogous predictive-stability analysis.


In [ ]:

KIDNEY_CLASSES = ["KICH", "KIRC", "KIRP"]
kidney_encoder = LabelEncoder().fit(KIDNEY_CLASSES)


def load_kidney_branch_panel_data(train_filename, valid_filename, genes):
    usecols = ["case_submitter_id", "class_label", *genes]
    train_df = pd.read_csv(KIDNEY_PROCESSED_DIR / train_filename, usecols=usecols)
    valid_df = pd.read_csv(KIDNEY_PROCESSED_DIR / valid_filename, usecols=usecols)
    return train_df, valid_df


kidney_hgb_rows = []
for branch_name, short_label, train_filename, valid_filename, genes in [
    ("primary_log2_tpm", "TPM", "primary_log2_tpm_training.csv",
     "primary_log2_tpm_heldout_validation.csv", kidney_tpm_panel),
    ("sensitivity_log2_cpm", "CPM", "sensitivity_log2_cpm_training.csv",
     "sensitivity_log2_cpm_heldout_validation.csv", kidney_cpm_panel),
]:
    train_df, valid_df = load_kidney_branch_panel_data(
        train_filename, valid_filename, genes
    )
    ytr = kidney_encoder.transform(train_df["class_label"].astype(str))
    yva = kidney_encoder.transform(valid_df["class_label"].astype(str))
    Xtr = train_df[genes].to_numpy(dtype=np.float32)
    Xva = valid_df[genes].to_numpy(dtype=np.float32)

    model = HistGradientBoostingClassifier(**FITNESS_MODEL_PARAMETERS)
    weights = compute_sample_weight(class_weight="balanced", y=ytr)
    with threadpool_limits(limits=1):
        model.fit(Xtr, ytr, sample_weight=weights)
        pred = model.predict(Xva)
        proba = model.predict_proba(Xva)

    original = kidney_performance.loc[kidney_performance["branch_name"] == branch_name].iloc[0]
    kidney_hgb_rows.append({
        "branch_name": branch_name,
        "representation": short_label,
        "selected_gene_count": len(genes),
        "selected_training_CV_balanced_accuracy_from_original_NSGA_run": original["nsga_training_cv_balanced_accuracy"],
        "heldout_accuracy_fixed_HGB": accuracy_score(yva, pred),
        "heldout_balanced_accuracy_fixed_HGB": balanced_accuracy_score(yva, pred),
        "heldout_macro_precision_fixed_HGB": precision_score(yva, pred, average="macro", zero_division=0),
        "heldout_macro_recall_fixed_HGB": recall_score(yva, pred, average="macro", zero_division=0),
        "heldout_macro_f1_fixed_HGB": f1_score(yva, pred, average="macro", zero_division=0),
        "heldout_macro_ovr_roc_auc_fixed_HGB": roc_auc_score(yva, proba, multi_class="ovr", average="macro"),
        "heldout_balanced_accuracy_original_CART": original["decision_tree_validation_balanced_accuracy"],
        "heldout_accuracy_original_CART": original["decision_tree_validation_accuracy"],
        "heldout_macro_ovr_roc_auc_original_CART": original["decision_tree_validation_macro_ovr_roc_auc"],
    })

kidney_fixed_hgb_performance = pd.DataFrame(kidney_hgb_rows)
kidney_fixed_hgb_performance.to_csv(
    KIDNEY_DIR / "Kidney_TPM_CPM_fixed_HGB_heldout_performance.csv", index=False
)
display(kidney_fixed_hgb_performance)

# Direct fixed-HGB analogue of the LGG predictive-stability figure.
fig, ax = plt.subplots(figsize=(7.0, 4.8))
x = np.arange(len(kidney_fixed_hgb_performance))
width = 0.25
ax.bar(x - width, kidney_fixed_hgb_performance["heldout_accuracy_fixed_HGB"], width, label="Accuracy")
ax.bar(x, kidney_fixed_hgb_performance["heldout_balanced_accuracy_fixed_HGB"], width, label="Balanced accuracy")
ax.bar(x + width, kidney_fixed_hgb_performance["heldout_macro_ovr_roc_auc_fixed_HGB"], width, label="Macro OVR ROC-AUC")
ax.set_xticks(x, kidney_fixed_hgb_performance["representation"])
ax.set_ylim(0, 1.02)
ax.set_ylabel("Held-out validation metric")
ax.set_title("Kidney Fixed-HGB Predictive Stability of TPM and CPM Panels")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "Kidney_TPM_CPM_fixed_HGB_predictive_stability", KIDNEY_DIR)
plt.show()
plt.close(fig)

# Balanced-accuracy comparison against the original post-selection CART models.
fig, ax = plt.subplots(figsize=(7.0, 4.8))
x = np.arange(len(kidney_fixed_hgb_performance))
width = 0.34
ax.bar(x - width/2, kidney_fixed_hgb_performance["heldout_balanced_accuracy_fixed_HGB"], width,
       label="Fixed HGB")
ax.bar(x + width/2, kidney_fixed_hgb_performance["heldout_balanced_accuracy_original_CART"], width,
       label="Original CART")
ax.set_xticks(x, kidney_fixed_hgb_performance["representation"])
ax.set_ylim(0, 1.02)
ax.set_ylabel("Held-out balanced accuracy")
ax.set_title("Kidney Predictive Performance by Model Family")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "Kidney_TPM_CPM_fixed_HGB_vs_CART_balanced_accuracy", KIDNEY_DIR)
plt.show()
plt.close(fig)

# Preserve the original CART summary plot as secondary context.
fig, ax = plt.subplots(figsize=(7.0, 4.8))
x = np.arange(len(kidney_performance))
width = 0.25
ax.bar(x - width, kidney_performance["decision_tree_validation_accuracy"], width, label="Accuracy")
ax.bar(x, kidney_performance["decision_tree_validation_balanced_accuracy"], width, label="Balanced accuracy")
ax.bar(x + width, kidney_performance["decision_tree_validation_macro_ovr_roc_auc"], width, label="Macro OVR ROC-AUC")
ax.set_xticks(x, ["TPM", "CPM"])
ax.set_ylim(0, 1.02)
ax.set_ylabel("Held-out validation metric")
ax.set_title("Kidney Original CART Predictive Performance")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "Kidney_TPM_CPM_original_CART_predictive_stability", KIDNEY_DIR)
plt.show()
plt.close(fig)


# Part III — LGG patient sampling perturbation with restartable checkpoints

This is the only section that launches new NSGA-II searches.

### Design

- Sampling universe: the original 197 LGG training patients only.
- Each replicate: stratified 80% subsample without replacement.
- Candidate gene universe: the same fixed original 3,000 genes for every replicate.
- Evolutionary seed: fixed at 42 for every replicate to reduce deliberate optimizer perturbation.
- Fitness model, operators, chromosome constraints, 10-fold CV, population size, and generation horizon: identical to the definitive Chapter 12 configuration in **publication mode**.
- The original 85 held-out patients remain excluded from all resampling NSGA-II fitness calculations.
- **Revision 3 checkpointing:** after every 25 completed generations, the complete `pymoo` algorithm state, including the current population, operator RNG states, problem state, and fitness cache, is written to a compressed checkpoint. The previous valid checkpoint is retained as a backup. On restart, the notebook resumes from the newest valid compatible checkpoint rather than restarting that replicate.

The checkpoint file is intentionally stored inside the replicate folder under `PAPER2_OUTPUTS`, so it persists across local-runtime or computer restarts. Completed replicates are detected and skipped as before.

**Important for the currently interrupted run:** Replicates 1–3 already have complete final outputs and will be skipped. The previously interrupted Replicate 4 was created by Revision 2, which did not save algorithm state. Therefore Replicate 4 must restart once from generation 1; after that, Revision 3 checkpoints protect it and all later replicates.


In [ ]:
RUN_RESAMPLING_NSGA = True
RESAMPLING_PROFILE = "PUBLICATION"   # "PILOT" or "PUBLICATION"

# Restartable checkpoint settings for Revision 3.
ENABLE_RESUMABLE_CHECKPOINTS = True
CHECKPOINT_EVERY_GENERATIONS = 25
CHECKPOINT_COMPRESSION_LEVEL = 1   # fast gzip; full fitness cache is retained
KEEP_PREVIOUS_CHECKPOINT_BACKUP = True

if RESAMPLING_PROFILE.upper() == "PILOT":
    RESAMPLE_REPLICATES = 2
    RESAMPLE_FRACTION = 0.80
    RESAMPLE_POPULATION = 30
    RESAMPLE_GENERATIONS = 5
elif RESAMPLING_PROFILE.upper() == "PUBLICATION":
    RESAMPLE_REPLICATES = 20
    RESAMPLE_FRACTION = 0.80
    RESAMPLE_POPULATION = 150
    RESAMPLE_GENERATIONS = 1000
else:
    raise ValueError("RESAMPLING_PROFILE must be PILOT or PUBLICATION")

print("RUN_RESAMPLING_NSGA =", RUN_RESAMPLING_NSGA)
print("Profile =", RESAMPLING_PROFILE)
print("Replicates =", RESAMPLE_REPLICATES)
print("Population =", RESAMPLE_POPULATION)
print("Generations =", RESAMPLE_GENERATIONS)
print("Restartable checkpoints =", ENABLE_RESUMABLE_CHECKPOINTS)
print("Checkpoint interval =", CHECKPOINT_EVERY_GENERATIONS, "completed generations")


## 23. Original controlled multicore NSGA-II machinery for the resampling experiment

This implementation follows the definitive Chapter 12 local multicore architecture: parallelization occurs across uncached chromosomes within a generation, while folds remain sequential inside each worker and inner BLAS/OpenMP threads are limited to one.

In [ ]:
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.callback import Callback
from pymoo.core.crossover import Crossover
from pymoo.core.problem import Problem
from pymoo.core.repair import Repair
from pymoo.core.sampling import Sampling
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.termination.max_gen import MaximumGenerationTermination
import pymoo

RESAMPLE_CV_SPLITS = 10
FITNESS_MODEL_SEED = 42
FIXED_EVOLUTIONARY_SEED_FOR_RESAMPLING = 42
CROSSOVER_PROBABILITY = 0.90
UNIFORM_GENE_SWAP_PROBABILITY = 0.50
MUTATION_PROBABILITY_PER_GENE = 0.01
MIN_GENES_PER_SOLUTION = 2
MAX_GENES_PER_SOLUTION = 50
INITIAL_MIN_GENES = 2
INITIAL_MAX_GENES = 20
PARETO_ACCURACY_TIE_TOLERANCE = 1e-8
ENABLE_FITNESS_CACHE = True
CHECKPOINT_FORMAT_VERSION = 1

RESAMPLE_FITNESS_MODEL_PARAMETERS = {
    "learning_rate": 0.01,
    "max_iter": 50,
    "max_depth": 5,
    "min_samples_leaf": 10,
    "max_bins": 20,
    "l2_regularization": 0.0,
    "early_stopping": False,
}


def soft_leanness(number_of_genes):
    return 1.0 / (math.sqrt(int(number_of_genes)) + 1.0)


def make_resample_fitness_classifier(fold_number):
    params = dict(RESAMPLE_FITNESS_MODEL_PARAMETERS)
    params["random_state"] = FITNESS_MODEL_SEED + int(fold_number)
    return HistGradientBoostingClassifier(**params)


def resample_cv_balanced_accuracy(X, y, selected_indices, cv_splits):
    selected_indices = np.asarray(selected_indices, dtype=int)
    if selected_indices.size == 0:
        return 0.0
    Xs = X[:, selected_indices]
    oof = np.full(y.shape, -1, dtype=int)
    for fold_number, (fit_idx, test_idx) in enumerate(cv_splits):
        model = make_resample_fitness_classifier(fold_number)
        w = compute_sample_weight(class_weight="balanced", y=y[fit_idx])
        model.fit(Xs[fit_idx], y[fit_idx], sample_weight=w)
        oof[test_idx] = model.predict(Xs[test_idx])
    return float(balanced_accuracy_score(y, oof))


def resample_fitness_worker(X, y, selected_indices, cv_splits):
    with threadpool_limits(limits=THREADS_PER_WORKER):
        ba = resample_cv_balanced_accuracy(X, y, selected_indices, cv_splits)
    return ba, soft_leanness(len(selected_indices)), os.getpid()


class SparseBinarySampling(Sampling):
    def __init__(self, minimum_genes, maximum_genes, seed):
        super().__init__()
        self.minimum_genes = int(minimum_genes)
        self.maximum_genes = int(maximum_genes)
        self.rng = np.random.default_rng(int(seed))
    def _do(self, problem, n_samples, *args, **kwargs):
        X = np.zeros((int(n_samples), int(problem.n_var)), dtype=bool)
        upper = min(self.maximum_genes, int(problem.n_var))
        lower = min(self.minimum_genes, upper)
        for row in range(int(n_samples)):
            count = int(self.rng.integers(lower, upper + 1))
            X[row, self.rng.choice(int(problem.n_var), size=count, replace=False)] = True
        return X


class UniformBinaryCrossover(Crossover):
    def __init__(self, probability=0.90, gene_swap_probability=0.50, seed=42):
        super().__init__(2, 2, prob=float(probability))
        self.gene_swap_probability = float(gene_swap_probability)
        self.rng = np.random.default_rng(int(seed))
    def _do(self, problem, X, **kwargs):
        X = np.asarray(X, dtype=bool)
        _, n_matings, n_var = X.shape
        mask = self.rng.random((n_matings, n_var)) < self.gene_swap_probability
        offspring = np.empty((2, n_matings, n_var), dtype=bool)
        offspring[0] = np.where(mask, X[1], X[0])
        offspring[1] = np.where(mask, X[0], X[1])
        return offspring


class FeatureCountRepair(Repair):
    def __init__(self, minimum_genes, maximum_genes, seed):
        super().__init__()
        self.minimum_genes = int(minimum_genes)
        self.maximum_genes = int(maximum_genes)
        self.rng = np.random.default_rng(int(seed))
    def _do(self, problem, X, **kwargs):
        X = np.asarray(X, dtype=bool).copy()
        maximum = min(self.maximum_genes, X.shape[1])
        minimum = min(self.minimum_genes, maximum)
        for row in range(X.shape[0]):
            selected = np.flatnonzero(X[row])
            if selected.size < minimum:
                unselected = np.flatnonzero(~X[row])
                add = self.rng.choice(unselected, size=minimum-selected.size, replace=False)
                X[row, add] = True
            selected = np.flatnonzero(X[row])
            if selected.size > maximum:
                remove = self.rng.choice(selected, size=selected.size-maximum, replace=False)
                X[row, remove] = False
        return X


class ResampleNSGAProblem(Problem):
    def __init__(self, X_train, y_train, cv_splits):
        self.X_train = np.asarray(X_train, dtype=np.float32, order="C")
        self.y_train = np.asarray(y_train, dtype=int)
        self.cv_splits = list(cv_splits)
        self.cache = {}
        self.total_requests = 0
        self.unique_evaluations = 0
        self.worker_pids = set()
        super().__init__(n_var=self.X_train.shape[1], n_obj=2, n_ieq_constr=0, xl=0, xu=1, vtype=bool)
    @staticmethod
    def _key(mask):
        return np.packbits(np.asarray(mask, dtype=np.uint8)).tobytes()
    def _evaluate(self, X, out, *args, **kwargs):
        chromosomes = np.atleast_2d(X).astype(bool, copy=False)
        self.total_requests += len(chromosomes)
        objective_rows = [None] * len(chromosomes)
        pending = {}
        for row_i, mask in enumerate(chromosomes):
            selected = np.flatnonzero(mask)
            if selected.size == 0:
                objective_rows[row_i] = np.array([1.0, 1.0])
                continue
            key = self._key(mask)
            if ENABLE_FITNESS_CACHE and key in self.cache:
                ba, lean = self.cache[key]
                objective_rows[row_i] = np.array([1-ba, 1-lean])
            else:
                pending.setdefault(key, {"selected": selected, "rows": []})["rows"].append(row_i)
        items = list(pending.items())
        if items:
            if len(items) >= 2 and N_WORKERS > 1:
                results = Parallel(
                    n_jobs=N_WORKERS, backend=JOBLIB_BACKEND, batch_size=1,
                    pre_dispatch="2*n_jobs", temp_folder=str(JOBLIB_TEMP_DIR),
                    max_nbytes="1M", mmap_mode="r"
                )(
                    delayed(resample_fitness_worker)(self.X_train, self.y_train, item[1]["selected"], self.cv_splits)
                    for item in items
                )
            else:
                results = [resample_fitness_worker(self.X_train, self.y_train, item[1]["selected"], self.cv_splits)
                           for item in items]
            for (key, item), (ba, lean, pid) in zip(items, results):
                self.unique_evaluations += 1
                self.worker_pids.add(int(pid))
                if ENABLE_FITNESS_CACHE:
                    self.cache[key] = (float(ba), float(lean))
                obj = np.array([1-ba, 1-lean], dtype=float)
                for row_i in item["rows"]:
                    objective_rows[row_i] = obj.copy()
        out["F"] = np.vstack(objective_rows)


def rank_pareto(pareto):
    return pareto.sort_values(
        ["cv_balanced_accuracy", "number_of_genes", "soft_leanness", "solution_id"],
        ascending=[False, True, False, True], kind="stable"
    ).reset_index(drop=True).assign(pareto_rank=lambda x: np.arange(1, len(x)+1))


def select_final_pareto(pareto):
    best = pareto["cv_balanced_accuracy"].max()
    candidates = pareto.loc[pareto["cv_balanced_accuracy"] >= best - PARETO_ACCURACY_TIE_TOLERANCE]
    return candidates.sort_values(
        ["number_of_genes", "soft_leanness", "solution_id"],
        ascending=[True, False, True], kind="stable"
    ).iloc[0]


class ResampleProgressCallback(Callback):
    def __init__(self, path):
        super().__init__()
        self.path = Path(path)
        self.data = {"generation": [], "best_population_balanced_accuracy": [],
                     "median_population_balanced_accuracy": [], "unique_model_evaluations": []}
    def notify(self, algorithm):
        F = np.asarray(algorithm.pop.get("F"), dtype=float)
        ba = 1 - F[:, 0]
        self.data["generation"].append(int(algorithm.n_gen))
        self.data["best_population_balanced_accuracy"].append(float(np.max(ba)))
        self.data["median_population_balanced_accuracy"].append(float(np.median(ba)))
        self.data["unique_model_evaluations"].append(int(algorithm.problem.unique_evaluations))
        if int(algorithm.n_gen) % 10 == 0:
            pd.DataFrame(self.data).to_csv(self.path, index=False)


def rebind_resample_callback(old_callback, progress_path, checkpoint_generation):
    """Attach a fresh callback object after loading a checkpoint in a new kernel/session.

    Jupyter/Colab can redefine notebook classes after a logout, reconnect, or kernel restart.
    A dill-loaded callback can therefore be behaviorally compatible while failing an
    isinstance(..., ResampleProgressCallback) test because it belongs to the prior
    session's dynamically defined class.  Always create a fresh callback from the
    current session and carry forward only the plain progress-data dictionary.
    """
    fresh = ResampleProgressCallback(progress_path)
    required = list(fresh.data.keys())
    source_df = None
    source_name = None

    old_data = getattr(old_callback, "data", None)
    if isinstance(old_data, dict) and all(k in old_data for k in required):
        try:
            lengths = [len(old_data[k]) for k in required]
            n = min(lengths) if lengths else 0
            source_df = pd.DataFrame({k: list(old_data[k])[:n] for k in required})
            source_name = "serialized checkpoint callback"
        except Exception:
            source_df = None

    # Fallback: use the progress CSV if the serialized callback's plain data cannot
    # be recovered.  Only rows at or before the restartable checkpoint are retained;
    # any newer progress rows describe work that was not checkpointed.
    if source_df is None and Path(progress_path).exists():
        try:
            candidate = pd.read_csv(progress_path)
            if all(k in candidate.columns for k in required):
                source_df = candidate[required].copy()
                source_name = "progress.csv"
        except Exception:
            source_df = None

    if source_df is not None and len(source_df):
        source_df = source_df.copy()
        source_df["generation"] = pd.to_numeric(source_df["generation"], errors="coerce")
        source_df = source_df.loc[
            source_df["generation"].notna()
            & (source_df["generation"].astype(int) <= int(checkpoint_generation))
        ]
        source_df = source_df.drop_duplicates(subset=["generation"], keep="last")
        source_df = source_df.sort_values("generation").reset_index(drop=True)
        fresh.data = {k: source_df[k].tolist() for k in required}
        print(
            f"  Reattached current-session progress callback using {source_name}; "
            f"restored {len(source_df)} progress rows through generation "
            f"{int(source_df['generation'].max()) if len(source_df) else 0}."
        )
    else:
        print(
            "  WARNING: checkpoint callback history could not be reconstructed. "
            "Evolutionary state is intact; progress logging will continue from the resumed generation."
        )

    return fresh


def candidate_pool_sha256(genes):
    return hashlib.sha256("\n".join(map(str, genes)).encode("utf-8")).hexdigest()


def checkpoint_paths(run_dir):
    run_dir = Path(run_dir)
    return (
        run_dir / "nsga_checkpoint.dill.gz",
        run_dir / "nsga_checkpoint.previous.dill.gz",
        run_dir / "checkpoint_status.json",
    )


def atomic_save_checkpoint(algorithm, checkpoint_path, backup_path, status_path, metadata, completed_generation):
    """Atomically save a complete restartable pymoo state after a completed generation."""
    checkpoint_path = Path(checkpoint_path)
    backup_path = Path(backup_path)
    status_path = Path(status_path)
    tmp_path = checkpoint_path.with_name(checkpoint_path.name + ".tmp")
    if tmp_path.exists():
        tmp_path.unlink()

    payload = {
        "checkpoint_format_version": CHECKPOINT_FORMAT_VERSION,
        "saved_completed_generation": int(completed_generation),
        "algorithm": algorithm,
        "python_random_state": random.getstate(),
        "numpy_random_state": np.random.get_state(),
        "metadata": dict(metadata),
    }

    # Compression level 1 is deliberately used to keep checkpoint pauses short.
    with open(tmp_path, "wb") as raw:
        with gzip.GzipFile(fileobj=raw, mode="wb", compresslevel=CHECKPOINT_COMPRESSION_LEVEL) as gz:
            dill.dump(payload, gz)
        raw.flush()
        os.fsync(raw.fileno())

    if checkpoint_path.exists():
        if KEEP_PREVIOUS_CHECKPOINT_BACKUP:
            os.replace(checkpoint_path, backup_path)
        else:
            checkpoint_path.unlink()
    os.replace(tmp_path, checkpoint_path)

    status = {
        **metadata,
        "checkpoint_format_version": CHECKPOINT_FORMAT_VERSION,
        "saved_completed_generation": int(completed_generation),
        "next_generation_on_resume": int(algorithm.n_gen),
        "checkpoint_file": str(checkpoint_path),
        "checkpoint_size_bytes": int(checkpoint_path.stat().st_size),
        "fitness_cache_entries": int(len(getattr(algorithm.problem, "cache", {}))),
        "unique_model_evaluations": int(getattr(algorithm.problem, "unique_evaluations", 0)),
        "pymoo_version": getattr(pymoo, "__version__", "unknown"),
        "python_version": platform.python_version(),
        "saved_unix_time": float(time.time()),
    }
    write_json(status, status_path)
    print(
        f"  Checkpoint saved after generation {completed_generation}: "
        f"{checkpoint_path.name} ({checkpoint_path.stat().st_size / (1024**2):.1f} MB)"
    )


def load_newest_valid_checkpoint(checkpoint_path, backup_path):
    """Load the primary checkpoint, falling back to the previous checkpoint if needed."""
    errors = []
    for candidate in [Path(checkpoint_path), Path(backup_path)]:
        if not candidate.exists():
            continue
        try:
            with gzip.open(candidate, "rb") as handle:
                payload = dill.load(handle)
            if not isinstance(payload, dict) or "algorithm" not in payload:
                raise ValueError("checkpoint payload is missing the algorithm state")
            if int(payload.get("checkpoint_format_version", -1)) != CHECKPOINT_FORMAT_VERSION:
                raise ValueError(
                    f"checkpoint format {payload.get('checkpoint_format_version')} is not supported "
                    f"by format {CHECKPOINT_FORMAT_VERSION}"
                )
            return payload, candidate, errors
        except Exception as exc:
            errors.append(f"{candidate}: {type(exc).__name__}: {exc}")
    return None, None, errors


def validate_checkpoint_compatibility(payload, expected_metadata):
    saved = payload.get("metadata", {})
    keys = [
        "replicate_id", "sampling_seed", "population_size", "target_generations",
        "subsample_n", "candidate_pool_n", "candidate_pool_sha256", "evolutionary_seed",
        "resample_fraction", "resample_cv_splits"
    ]
    mismatches = []
    for key in keys:
        if saved.get(key) != expected_metadata.get(key):
            mismatches.append((key, saved.get(key), expected_metadata.get(key)))
    saved_pymoo = saved.get("pymoo_version")
    current_pymoo = getattr(pymoo, "__version__", "unknown")
    if saved_pymoo is not None and saved_pymoo != current_pymoo:
        mismatches.append(("pymoo_version", saved_pymoo, current_pymoo))
    if mismatches:
        lines = [f"  {k}: checkpoint={a!r}, current={b!r}" for k, a, b in mismatches]
        raise RuntimeError(
            "A checkpoint exists but is incompatible with the current run settings:\n" + "\n".join(lines) +
            "\nRename or remove the checkpoint files only if you intentionally want to restart this replicate."
        )


def archive_progress_file(progress_path, archive_path):
    progress_path = Path(progress_path)
    archive_path = Path(archive_path)
    if progress_path.exists() and not archive_path.exists():
        shutil.copy2(progress_path, archive_path)
        print("  Archived prior diagnostic progress to:", archive_path.name)


print("Resampling NSGA-II functions and restartable checkpoint helpers loaded.")


## 24. Run or resume LGG patient sampling perturbation searches

For each incomplete replicate, this cell checks for a valid primary checkpoint and then for its backup. If one is found and its run metadata match the current settings, the saved algorithm state is restored and evolution continues from the next generation. If no checkpoint exists, the replicate starts from generation 1. Fully completed replicates remain untouched and are skipped.

`progress.csv` is still written every 10 generations for monitoring. Because progress rows do not contain the evolutionary population, any rows beyond the most recent checkpoint cannot themselves be resumed. Revision 3 archives such uncheckpointed diagnostic progress before continuing from the checkpoint.


In [ ]:
def run_one_patient_resample(replicate_id, sampling_seed, population_size, generations):
    profile_dir = RESAMPLE_DIR / RESAMPLING_PROFILE.upper()
    run_dir = profile_dir / f"replicate_{replicate_id:02d}"
    run_dir.mkdir(parents=True, exist_ok=True)

    selected_path = run_dir / "selected_pareto_solution.csv"
    genes_path = run_dir / "selected_genes.csv"
    summary_path = run_dir / "run_summary.json"
    patients_path = run_dir / "subsampled_patients.csv"
    progress_path = run_dir / "progress.csv"
    checkpoint_path, checkpoint_backup_path, checkpoint_status_path = checkpoint_paths(run_dir)

    if all(p.exists() for p in [selected_path, genes_path, summary_path, patients_path]):
        print(f"Replicate {replicate_id:02d}: complete outputs found; skipping.")
        return json.loads(summary_path.read_text(encoding="utf-8"))

    # Labels in the exact lgg_train row order.
    labels = split_labels.loc[lgg_train["case_submitter_id"]].to_numpy()
    enc = LabelEncoder().fit(LGG_CLASSES)
    y_all = enc.transform(labels)
    all_indices = np.arange(len(lgg_train))
    sub_idx, _ = train_test_split(
        all_indices,
        train_size=RESAMPLE_FRACTION,
        stratify=y_all,
        random_state=int(sampling_seed),
    )
    sub_idx = np.sort(sub_idx)
    X_sub = lgg_train.iloc[sub_idx][lgg_candidate_genes].to_numpy(dtype=np.float32)
    y_sub = y_all[sub_idx]
    sub_ids = lgg_train.iloc[sub_idx]["case_submitter_id"].astype(str).tolist()

    # Save the deterministic patient subsample immediately rather than waiting for completion.
    current_patients = pd.DataFrame({
        "case_submitter_id": sub_ids,
        "molecular_subtype": labels[sub_idx],
    })
    if patients_path.exists():
        previous_patients = pd.read_csv(patients_path)
        if previous_patients["case_submitter_id"].astype(str).tolist() != sub_ids:
            raise RuntimeError(
                f"Replicate {replicate_id:02d}: saved patient subsample does not match sampling seed {sampling_seed}."
            )
    else:
        current_patients.to_csv(patients_path, index=False)

    cv = StratifiedKFold(n_splits=RESAMPLE_CV_SPLITS, shuffle=True, random_state=42)
    cv_splits = list(cv.split(X_sub, y_sub))

    evo_seed = FIXED_EVOLUTIONARY_SEED_FOR_RESAMPLING
    expected_metadata = {
        "replicate_id": int(replicate_id),
        "sampling_seed": int(sampling_seed),
        "resample_fraction": float(RESAMPLE_FRACTION),
        "subsample_n": int(len(sub_idx)),
        "candidate_pool_n": int(len(lgg_candidate_genes)),
        "candidate_pool_sha256": candidate_pool_sha256(lgg_candidate_genes),
        "evolutionary_seed": int(evo_seed),
        "population_size": int(population_size),
        "target_generations": int(generations),
        "resample_cv_splits": int(RESAMPLE_CV_SPLITS),
        "pymoo_version": getattr(pymoo, "__version__", "unknown"),
        "notebook_revision": NOTEBOOK_REVISION,
    }

    resumed_from_checkpoint = False
    resumed_checkpoint_file = None
    checkpoint_start_generation = 0

    payload = None
    payload_source = None
    checkpoint_errors = []
    if ENABLE_RESUMABLE_CHECKPOINTS:
        payload, payload_source, checkpoint_errors = load_newest_valid_checkpoint(
            checkpoint_path, checkpoint_backup_path
        )
        for msg in checkpoint_errors:
            print("  Checkpoint load warning:", msg)

    if payload is not None:
        validate_checkpoint_compatibility(payload, expected_metadata)
        algorithm = payload["algorithm"]
        problem = algorithm.problem
        old_callback = getattr(algorithm, "callback", None)
        checkpoint_start_generation = int(payload["saved_completed_generation"])

        # A Colab logout/reconnect or Jupyter kernel restart can redefine notebook
        # classes.  The dill-loaded callback can then fail isinstance() even though
        # the checkpoint itself is valid.  Rebind a fresh current-session callback
        # and preserve its plain historical progress data instead of rejecting the
        # checkpoint on Python class identity alone.
        callback = rebind_resample_callback(
            old_callback, progress_path, checkpoint_start_generation
        )
        algorithm.callback = callback

        if problem.X_train.shape != X_sub.shape or not np.array_equal(problem.y_train, y_sub):
            raise RuntimeError(
                f"Replicate {replicate_id:02d}: checkpoint problem data do not match the deterministic subsample."
            )

        # Restore Python/NumPy global RNG states in addition to the RNG state contained
        # inside the serialized pymoo algorithm and custom operators.
        if payload.get("python_random_state") is not None:
            random.setstate(payload["python_random_state"])
        if payload.get("numpy_random_state") is not None:
            np.random.set_state(payload["numpy_random_state"])

        callback.path = progress_path
        algorithm.callback = callback
        problem.worker_pids = set()  # report workers observed during this resumed process only

        resumed_from_checkpoint = True
        resumed_checkpoint_file = str(payload_source)

        # Progress can be newer than the latest checkpoint because it is written every
        # 10 generations while restartable state is saved every 25. Preserve that
        # diagnostic tail before the callback overwrites progress.csv during resume.
        if progress_path.exists():
            try:
                old_progress = pd.read_csv(progress_path)
                old_max = int(old_progress["generation"].max()) if len(old_progress) else 0
            except Exception:
                old_max = 0
            if old_max > checkpoint_start_generation:
                archive_progress_file(
                    progress_path,
                    run_dir / f"progress_uncheckpointed_tail_after_gen_{checkpoint_start_generation:04d}.csv"
                )

        print("="*78)
        print(f"Patient resample {replicate_id:02d}/{RESAMPLE_REPLICATES} | sampling seed={sampling_seed}")
        print(
            f"RESUMING from {Path(payload_source).name}: "
            f"generation {checkpoint_start_generation} completed; next generation={algorithm.n_gen}"
        )
        print(f"n={len(sub_idx)}, population={population_size}, target generations={generations}, workers={N_WORKERS}")
    else:
        # Revision-2 progress files contain monitoring information only and cannot
        # reconstruct the population. Archive them once before the fresh restart.
        if progress_path.exists():
            archive_progress_file(
                progress_path,
                run_dir / "progress_legacy_incomplete_pre_checkpoint.csv"
            )

        random.seed(evo_seed)
        np.random.seed(evo_seed)
        problem = ResampleNSGAProblem(X_sub, y_sub, cv_splits)
        callback = ResampleProgressCallback(progress_path)
        algorithm = NSGA2(
            pop_size=int(population_size),
            sampling=SparseBinarySampling(INITIAL_MIN_GENES, INITIAL_MAX_GENES, evo_seed),
            crossover=UniformBinaryCrossover(
                CROSSOVER_PROBABILITY, UNIFORM_GENE_SWAP_PROBABILITY, evo_seed+11
            ),
            mutation=BitflipMutation(prob=1.0, prob_var=MUTATION_PROBABILITY_PER_GENE),
            repair=FeatureCountRepair(MIN_GENES_PER_SOLUTION, MAX_GENES_PER_SOLUTION, evo_seed+1),
            eliminate_duplicates=True,
        )
        algorithm.setup(
            problem,
            termination=("n_gen", int(generations)),
            seed=evo_seed,
            callback=callback,
            save_history=False,
            verbose=False,
        )

        print("="*78)
        print(f"Patient resample {replicate_id:02d}/{RESAMPLE_REPLICATES} | sampling seed={sampling_seed}")
        if progress_path.exists():
            print("No restartable checkpoint found; starting this incomplete replicate from generation 1.")
        print(f"n={len(sub_idx)}, population={population_size}, generations={generations}, workers={N_WORKERS}")

    # A loaded checkpoint already contains all evolutionary state. Replace only the
    # termination target so the serialized search continues to the requested horizon.
    if resumed_from_checkpoint:
        algorithm.termination = MaximumGenerationTermination(int(generations))

    start = time.perf_counter()
    if resumed_from_checkpoint and checkpoint_start_generation >= int(generations):
        print("Checkpoint already contains the target generation; reconstructing final outputs without new evolution.")
    else:
        try:
            while algorithm.has_next():
                algorithm.next()
                # pymoo increments n_gen after finishing a generation, so n_gen - 1 is
                # the generation whose population has just been fully evaluated/selected.
                completed_generation = int(algorithm.n_gen) - 1
                if (
                    ENABLE_RESUMABLE_CHECKPOINTS
                    and completed_generation > 0
                    and completed_generation % int(CHECKPOINT_EVERY_GENERATIONS) == 0
                ):
                    atomic_save_checkpoint(
                        algorithm,
                        checkpoint_path,
                        checkpoint_backup_path,
                        checkpoint_status_path,
                        expected_metadata,
                        completed_generation,
                    )
        except BaseException:
            print(
                "Run interrupted. The most recent completed atomic checkpoint remains on disk; "
                "rerun this cell/notebook to resume from it."
            )
            raise

    elapsed_this_session = time.perf_counter() - start
    result = algorithm.result()
    problem = algorithm.problem
    callback = algorithm.callback

    chromosomes = np.atleast_2d(result.X).astype(bool)
    objectives = np.atleast_2d(result.F).astype(float)
    records = []
    for solution_id, (mask, F) in enumerate(zip(chromosomes, objectives), start=1):
        genes = [lgg_candidate_genes[i] for i in np.flatnonzero(mask)]
        records.append({
            "solution_id": solution_id,
            "cv_balanced_accuracy": 1-F[0],
            "soft_leanness": 1-F[1],
            "number_of_genes": int(mask.sum()),
            "selected_gene_ids": ";".join(genes),
            "selected_gene_symbols": ";".join(gene_symbol(g, lgg_gene_to_symbol) for g in genes),
        })
    pareto = rank_pareto(pd.DataFrame(records).drop_duplicates())
    selected = select_final_pareto(pareto)
    selected_genes = split_semicolon(selected["selected_gene_ids"])

    pareto.to_csv(run_dir / "final_pareto_front.csv", index=False)
    pd.DataFrame([selected]).to_csv(selected_path, index=False)
    pd.DataFrame({
        "gene_id": selected_genes,
        "gene_name": [gene_symbol(g, lgg_gene_to_symbol) for g in selected_genes]
    }).to_csv(genes_path, index=False)
    current_patients.to_csv(patients_path, index=False)
    pd.DataFrame(callback.data).to_csv(progress_path, index=False)

    summary = {
        "replicate_id": int(replicate_id),
        "sampling_seed": int(sampling_seed),
        "subsample_fraction": RESAMPLE_FRACTION,
        "subsample_n": int(len(sub_idx)),
        "candidate_pool_fixed_n": len(lgg_candidate_genes),
        "evolutionary_seed_fixed": evo_seed,
        "population_size": int(population_size),
        "generations": int(generations),
        "selected_cv_balanced_accuracy": float(selected["cv_balanced_accuracy"]),
        "selected_gene_count": int(selected["number_of_genes"]),
        "elapsed_hours_this_session": float(elapsed_this_session/3600),
        "unique_model_evaluations": int(problem.unique_evaluations),
        "observed_worker_processes_this_session": len(problem.worker_pids),
        "resumed_from_checkpoint": bool(resumed_from_checkpoint),
        "checkpoint_start_generation": int(checkpoint_start_generation),
        "resumed_checkpoint_file": resumed_checkpoint_file,
        "checkpoint_interval_generations": int(CHECKPOINT_EVERY_GENERATIONS),
        "checkpointing_enabled": bool(ENABLE_RESUMABLE_CHECKPOINTS),
    }
    write_json(summary, summary_path)
    print("Completed:", summary)
    return summary


if RUN_RESAMPLING_NSGA:
    sampling_seeds = [10_000 + i for i in range(1, RESAMPLE_REPLICATES + 1)]
    resample_summaries = []
    for rep_id, sampling_seed in enumerate(sampling_seeds, start=1):
        resample_summaries.append(run_one_patient_resample(
            rep_id, sampling_seed, RESAMPLE_POPULATION, RESAMPLE_GENERATIONS
        ))
    pd.DataFrame(resample_summaries).to_csv(
        RESAMPLE_DIR / RESAMPLING_PROFILE.upper() / "all_resample_run_summaries.csv", index=False
    )
else:
    print("Patient-resampling NSGA-II is disabled. Stage I analyses are complete without this step.")


## 25. Analyze completed patient sampling perturbation runs

This cell can be rerun at any time. It analyzes whatever completed replicates are present under the selected profile and compares their stability with the original five evolutionary-seed runs.

In [ ]:
def load_completed_resample_panels(profile):
    profile_dir = RESAMPLE_DIR / profile.upper()
    panels = {}
    summaries = []
    if not profile_dir.exists():
        return panels, pd.DataFrame()
    for run_dir in sorted(profile_dir.glob("replicate_*")):
        genes_path = run_dir / "selected_genes.csv"
        summary_path = run_dir / "run_summary.json"
        if genes_path.exists() and summary_path.exists():
            rep = int(run_dir.name.split("_")[-1])
            panels[rep] = pd.read_csv(genes_path)["gene_id"].astype(str).tolist()
            summaries.append(json.loads(summary_path.read_text(encoding="utf-8")))
    return panels, pd.DataFrame(summaries)

resample_panels, resample_run_summary = load_completed_resample_panels(RESAMPLING_PROFILE)
print(f"Completed {RESAMPLING_PROFILE} resampling panels found: {len(resample_panels)}")

if len(resample_panels) >= 2:
    Z_resample, _ = selection_matrix(resample_panels, lgg_candidate_genes)
    resample_exact = pairwise_exact_stability(resample_panels, len(lgg_candidate_genes))
    resample_phi = nogueira_stability(Z_resample)

    resample_corr_rows = []
    for a, b in combinations(resample_panels.keys(), 2):
        sm, _ = panel_match(
            resample_panels[a], resample_panels[b],
            lgg_gene_to_index, lgg_gene_to_index,
            lgg_abs_corr, PRIMARY_CORRELATION_THRESHOLD
        )
        resample_corr_rows.append({"replicate_a": a, "replicate_b": b, **sm})
    resample_corr = pd.DataFrame(resample_corr_rows)

    profile_dir = RESAMPLE_DIR / RESAMPLING_PROFILE.upper()
    resample_exact.to_csv(profile_dir / "resample_pairwise_exact_stability.csv", index=False)
    resample_corr.to_csv(profile_dir / "resample_pairwise_correlation_adjusted_stability.csv", index=False)
    resample_run_summary.to_csv(profile_dir / "all_resample_run_summaries_loaded.csv", index=False)

    comparison = pd.DataFrame([
        {
            "perturbation_type": "evolutionary_seed_same_197_training_patients",
            "runs": len(lgg_panels),
            "nogueira_stability": lgg_phi,
            "mean_exact_jaccard": lgg_exact_pairwise["jaccard"].mean(),
            "mean_correlation_adjusted_jaccard_tau_0.80": observed_primary["correlation_adjusted_jaccard"].mean(),
            "mean_matched_coverage_tau_0.80": observed_primary["matched_coverage_smaller_panel"].mean(),
        },
        {
            "perturbation_type": "patient_subsample_fixed_candidate_pool_and_evolutionary_seed",
            "runs": len(resample_panels),
            "nogueira_stability": resample_phi,
            "mean_exact_jaccard": resample_exact["jaccard"].mean(),
            "mean_correlation_adjusted_jaccard_tau_0.80": resample_corr["correlation_adjusted_jaccard"].mean(),
            "mean_matched_coverage_tau_0.80": resample_corr["matched_coverage_smaller_panel"].mean(),
        },
    ])
    comparison.to_csv(profile_dir / "optimization_vs_sampling_stability_comparison.csv", index=False)
    display(comparison)
else:
    print("At least two completed resampling runs are required for stability analysis.")

# Part IV — Consolidated Paper 2 outputs from the core analyses


In [ ]:
# LGG primary-threshold summary.
lgg_primary_chance = lgg_corr_adjusted_chance.loc[
    np.isclose(lgg_corr_adjusted_chance["threshold"], PRIMARY_CORRELATION_THRESHOLD)
].copy()

lgg_primary_summary = lgg_primary_chance.agg({
    "exact_jaccard": ["mean", "median", "min", "max"],
    "correlation_adjusted_jaccard": ["mean", "median", "min", "max"],
    "chance_corrected_corr_adjusted_stability": ["mean", "median", "min", "max"],
    "matched_coverage_smaller_panel": ["mean", "median", "min", "max"],
    "matched_count": ["mean", "median", "min", "max"],
}).T.reset_index().rename(columns={"index": "metric"})
lgg_primary_summary.to_csv(TABLE_DIR / "Paper2_LGG_primary_threshold_stability_summary.csv", index=False)

kidney_primary_chance = kidney_corr_adjusted_chance.loc[
    np.isclose(kidney_corr_adjusted_chance["threshold"], PRIMARY_CORRELATION_THRESHOLD)
].copy()
kidney_primary_summary = kidney_primary_chance[[
    "representation", "exact_selected_panel_jaccard", "matched_count",
    "matched_coverage_smaller_panel", "correlation_adjusted_jaccard",
    "null_mean_corr_adjusted_jaccard", "chance_corrected_corr_adjusted_stability",
    "mean_abs_correlation_of_matches"
]].copy()
kidney_primary_summary.to_csv(TABLE_DIR / "Paper2_Kidney_primary_threshold_stability_summary.csv", index=False)

lgg_primary_cc_mean = float(lgg_primary_chance["chance_corrected_corr_adjusted_stability"].mean())
kidney_consensus_primary = kidney_primary_chance.loc[
    kidney_primary_chance["representation"] == "consensus_min_abs_r"
].iloc[0]

key_results = pd.DataFrame([
    {
        "dataset_experiment": "TCGA-LGG five evolutionary seeds",
        "perturbation": "optimization stochasticity",
        "common_candidate_universe": True,
        "candidate_universe_size": 3000,
        "number_of_panels": 5,
        "panel_size": 50,
        "unique_selected_genes": len(set().union(*map(set, lgg_panels.values()))),
        "mean_exact_jaccard": lgg_exact_pairwise["jaccard"].mean(),
        "nogueira_stability": lgg_phi,
        "mean_corr_adjusted_jaccard_tau_0.80": observed_primary["correlation_adjusted_jaccard"].mean(),
        "mean_chance_corrected_corr_adjusted_stability_tau_0.80": lgg_primary_cc_mean,
        "global_null_largest_component_size": lgg_global_observed_metrics["largest_component_size"],
        "global_null_large_components_seeds_ge4": lgg_global_observed_metrics["large_components_seeds_ge4"],
        "mean_fixed_HGB_heldout_balanced_accuracy": lgg_predictive_reproducibility["heldout_balanced_accuracy_fixed_HGB"].mean(),
    },
    {
        "dataset_experiment": "TCGA-Kidney TPM versus CPM",
        "perturbation": "expression representation / preprocessing branch",
        "common_candidate_universe": False,
        "candidate_universe_size": np.nan,
        "number_of_panels": 2,
        "panel_size": 50,
        "unique_selected_genes": len(set(kidney_tpm_panel) | set(kidney_cpm_panel)),
        "mean_exact_jaccard": jaccard(kidney_tpm_panel, kidney_cpm_panel),
        "nogueira_stability": np.nan,
        "mean_corr_adjusted_jaccard_tau_0.80": kidney_consensus_primary["correlation_adjusted_jaccard"],
        "mean_chance_corrected_corr_adjusted_stability_tau_0.80": kidney_consensus_primary["chance_corrected_corr_adjusted_stability"],
        "global_null_largest_component_size": np.nan,
        "global_null_large_components_seeds_ge4": np.nan,
        "mean_fixed_HGB_heldout_balanced_accuracy": kidney_fixed_hgb_performance["heldout_balanced_accuracy_fixed_HGB"].mean(),
    },
])
key_results.to_csv(TABLE_DIR / "Paper2_key_results_summary.csv", index=False)
display(key_results)

analysis_config = {
    "notebook_revision": NOTEBOOK_REVISION,
    "base_directory": BASE_DIR,
    "LGG_input_directory": LGG_DIR,
    "kidney_processed_input_directory": KIDNEY_PROCESSED_DIR,
    "kidney_modeling_input_directory": KIDNEY_MODELING_DIR,
    "workers": N_WORKERS,
    "threads_per_worker": THREADS_PER_WORKER,
    "correlation_thresholds": CORRELATION_THRESHOLDS,
    "primary_correlation_threshold": PRIMARY_CORRELATION_THRESHOLD,
    "null_replicates": N_NULL_REPLICATES,
    "global_five_panel_null_replicates": N_NULL_REPLICATES,
    "correlation_bootstraps": N_CORRELATION_BOOTSTRAPS,
    "LGG_seeds": LGG_SEEDS,
    "LGG_original_analysis_configuration": lgg_original_config,
    "chance_correction_definition": "(observed - mean_null) / (1 - mean_null)",
    "global_null_design": "five independent 50-gene panels from fixed 3000-gene LGG candidate universe",
    "resampling_enabled_in_this_run": RUN_RESAMPLING_NSGA,
    "resampling_profile": RESAMPLING_PROFILE,
    "resampling_candidate_pool_scope": "fixed original 3000-gene candidate pool",
    "resampling_restartable_checkpoints": ENABLE_RESUMABLE_CHECKPOINTS,
    "resampling_checkpoint_interval_generations": CHECKPOINT_EVERY_GENERATIONS,
    "resampling_checkpoint_compression_level": CHECKPOINT_COMPRESSION_LEVEL,
    "resampling_checkpoint_backup_retained": KEEP_PREVIOUS_CHECKPOINT_BACKUP,
}
write_json(analysis_config, OUTPUT_DIR / "Paper2_analysis_configuration.json")

print("Core Paper 2 analyses complete.")
print("Main output directory:", OUTPUT_DIR)


# Part V — Final analysis of all 20 publication patient resamples

This section is deliberately self-contained and **does not rerun NSGA-II**. It uses the completed `PUBLICATION/replicate_01` through `replicate_20` outputs to perform the final lightweight analyses needed for Paper 2:

1. held-out predictive evaluation of each panel when the HGB classifier is fit on its original 157-patient resample;
2. panel-only evaluation after refitting the same HGB classifier on all 197 original training patients;
3. running stability estimates using the first 5, 10, 15, and 20 resamples; and
4. formal gene-recurrence tables across all 20 resamples.

The untouched 85-patient held-out validation cohort remains the evaluation cohort for both predictive designs.


In [ ]:
# ============================================================
# FINAL PAPER 2 POST-ANALYSIS OF THE 20 PUBLICATION RESAMPLES
# ============================================================
# This cell is intentionally self-contained. It does NOT rerun NSGA-II.
# It uses the 20 completed publication resampling panels to evaluate:
#   1) held-out predictive generalization under two training designs;
#   2) stability convergence using the first 5, 10, 15, and 20 resamples;
#   3) formal gene recurrence across all 20 resamples.

from pathlib import Path
from itertools import combinations
from collections import Counter
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from scipy.optimize import linear_sum_assignment
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_sample_weight
from threadpoolctl import threadpool_limits

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
POST_DEFAULT_BASE_DIR = Path(
    r"C:\Users\we3mo\Dropbox\MY DATA DRIVE\CANCER"
    r"\2026 - ADD ANALYSIS FOR TCGA BIOINFORMATICS_2"
)
POST_BASE_DIR = Path(os.environ.get("PAPER2_BASE_DIR", str(POST_DEFAULT_BASE_DIR)))
POST_LGG_REQUESTED = POST_BASE_DIR / "TCGA_LGG_MOLECULAR_SUBTYPE_FINAL_ANALYSIS"
POST_OUTPUT_DIR = POST_BASE_DIR / "PAPER2_OUTPUTS"
POST_RESAMPLE_DIR = POST_OUTPUT_DIR / "09_SAMPLE_PERTURBATION" / "PUBLICATION"
POST_RESULTS_DIR = POST_OUTPUT_DIR / "12_RESAMPLING_GENERALIZATION"
POST_PREDICTIVE_DIR = POST_OUTPUT_DIR / "11_PREDICTIVE_STABILITY"
POST_TABLE_DIR = POST_OUTPUT_DIR / "TABLES"
POST_FIGURE_DIR = POST_OUTPUT_DIR / "FIGURES"
POST_CACHE_DIR = POST_OUTPUT_DIR / "CACHE"

for directory in [
    POST_RESULTS_DIR,
    POST_PREDICTIVE_DIR,
    POST_TABLE_DIR,
    POST_FIGURE_DIR,
    POST_CACHE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def _post_resolve_root(requested_root, required_filename):
    requested_root = Path(requested_root)
    if (requested_root / required_filename).exists():
        return requested_root
    if not requested_root.exists():
        raise FileNotFoundError(f"Input directory not found: {requested_root}")
    candidates = [
        p for p in requested_root.iterdir()
        if p.is_dir() and (p / required_filename).exists()
    ]
    if len(candidates) == 1:
        return candidates[0]
    raise FileNotFoundError(
        f"Could not locate {required_filename} under {requested_root}. "
        f"Nested candidates found: {[str(x) for x in candidates]}"
    )


POST_LGG_DIR = _post_resolve_root(POST_LGG_REQUESTED, "patient_split_assignments.csv")

required_post_files = [
    POST_LGG_DIR / "patient_split_assignments.csv",
    POST_LGG_DIR / "NSGA_candidate_gene_pool.csv",
    POST_LGG_DIR / "training_candidate_expression_log2TPM1.csv.gz",
    POST_LGG_DIR / "heldout_validation_candidate_expression_log2TPM1.csv.gz",
]
missing_post_files = [str(p) for p in required_post_files if not p.exists()]
if missing_post_files:
    raise FileNotFoundError("Missing required LGG input files:\n" + "\n".join(missing_post_files))
if not POST_RESAMPLE_DIR.exists():
    raise FileNotFoundError(f"Publication resampling directory not found: {POST_RESAMPLE_DIR}")

# ------------------------------------------------------------
# Load fixed LGG data and all completed publication panels
# ------------------------------------------------------------
post_split = pd.read_csv(POST_LGG_DIR / "patient_split_assignments.csv")
post_candidates = pd.read_csv(POST_LGG_DIR / "NSGA_candidate_gene_pool.csv")
post_train = pd.read_csv(POST_LGG_DIR / "training_candidate_expression_log2TPM1.csv.gz")
post_valid = pd.read_csv(POST_LGG_DIR / "heldout_validation_candidate_expression_log2TPM1.csv.gz")

post_candidate_genes = post_candidates["gene_id"].astype(str).tolist()
post_gene_to_symbol = dict(zip(
    post_candidates["gene_id"].astype(str),
    post_candidates["gene_name"].astype(str),
))
post_gene_to_index = {g: i for i, g in enumerate(post_candidate_genes)}

post_panels = {}
post_subsamples = {}
post_run_summaries = []

for run_dir in sorted(POST_RESAMPLE_DIR.glob("replicate_*")):
    genes_path = run_dir / "selected_genes.csv"
    patients_path = run_dir / "subsampled_patients.csv"
    summary_path = run_dir / "run_summary.json"
    if genes_path.exists() and patients_path.exists() and summary_path.exists():
        replicate_id = int(run_dir.name.split("_")[-1])
        post_panels[replicate_id] = pd.read_csv(genes_path)["gene_id"].astype(str).tolist()
        post_subsamples[replicate_id] = pd.read_csv(patients_path)
        post_run_summaries.append(json.loads(summary_path.read_text(encoding="utf-8")))

post_replicate_ids = sorted(post_panels)
print(f"Completed publication resamples found: {len(post_replicate_ids)}")
print("Replicate IDs:", post_replicate_ids)

if len(post_replicate_ids) != 20:
    raise RuntimeError(
        f"This final publication post-analysis expects 20 completed resamples; "
        f"found {len(post_replicate_ids)}."
    )

if post_replicate_ids != list(range(1, 21)):
    raise RuntimeError(f"Expected replicate IDs 1-20; found {post_replicate_ids}")

for rep in post_replicate_ids:
    if not set(post_panels[rep]).issubset(set(post_candidate_genes)):
        raise RuntimeError(f"Replicate {rep:02d} contains genes outside the fixed candidate pool.")
    if len(post_subsamples[rep]) != 157:
        raise RuntimeError(
            f"Replicate {rep:02d} has {len(post_subsamples[rep])} subsampled patients; expected 157."
        )

post_run_summary_df = pd.DataFrame(post_run_summaries).sort_values("replicate_id")
post_summary_by_rep = {
    int(row["replicate_id"]): row
    for row in post_run_summaries
}

# ------------------------------------------------------------
# Fixed classifier and label definitions
# ------------------------------------------------------------
POST_LGG_CLASSES = ["IDHmut-codel", "IDHmut-non-codel", "IDHwt"]
post_class_to_int = {label: i for i, label in enumerate(POST_LGG_CLASSES)}
post_label_by_id = post_split.set_index("case_submitter_id")["molecular_subtype"].to_dict()

post_y_train_all = np.asarray([
    post_class_to_int[post_label_by_id[cid]]
    for cid in post_train["case_submitter_id"].astype(str)
], dtype=int)
post_y_valid = np.asarray([
    post_class_to_int[post_label_by_id[cid]]
    for cid in post_valid["case_submitter_id"].astype(str)
], dtype=int)

post_train_id_to_row = {
    cid: i for i, cid in enumerate(post_train["case_submitter_id"].astype(str))
}
post_valid_ids = post_valid["case_submitter_id"].astype(str).tolist()

POST_HGB_PARAMETERS = {
    "learning_rate": 0.01,
    "max_iter": 50,
    "max_depth": 5,
    "min_samples_leaf": 10,
    "max_bins": 20,
    "l2_regularization": 0.0,
    "early_stopping": False,
    "random_state": 42,
}


def _post_classification_metrics(y_true, pred, proba):
    values = {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "macro_precision": precision_score(y_true, pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
    }
    try:
        values["macro_ovr_roc_auc"] = roc_auc_score(
            y_true, proba, multi_class="ovr", average="macro"
        )
    except ValueError:
        values["macro_ovr_roc_auc"] = np.nan
    return values


# ============================================================
# A. HELD-OUT PREDICTIVE GENERALIZATION OF ALL 20 PANELS
# ============================================================
# Design 1: Sampling-pipeline evaluation
#   Fit on the exact 157-patient subsample that generated the panel.
# Design 2: Panel-only evaluation
#   Hold the selected panel fixed, refit the same HGB model on all 197 original
#   training patients, and evaluate on the same untouched 85 held-out patients.

post_performance_rows = []
post_prediction_rows = []

for rep in post_replicate_ids:
    genes = post_panels[rep]
    subsample_ids = post_subsamples[rep]["case_submitter_id"].astype(str).tolist()

    missing_ids = [cid for cid in subsample_ids if cid not in post_train_id_to_row]
    if missing_ids:
        raise RuntimeError(
            f"Replicate {rep:02d} contains patient IDs not found in the original 197-patient training cohort."
        )

    subsample_rows = np.asarray([post_train_id_to_row[cid] for cid in subsample_ids], dtype=int)
    all_training_rows = np.arange(len(post_train), dtype=int)
    X_valid_panel = post_valid[genes].to_numpy(dtype=np.float32)

    evaluation_designs = {
        "sampling_pipeline_157_patients": subsample_rows,
        "panel_only_refit_197_patients": all_training_rows,
    }

    for design, train_rows in evaluation_designs.items():
        X_fit = post_train.iloc[train_rows][genes].to_numpy(dtype=np.float32)
        y_fit = post_y_train_all[train_rows]
        weights = compute_sample_weight(class_weight="balanced", y=y_fit)

        model = HistGradientBoostingClassifier(**POST_HGB_PARAMETERS)
        with threadpool_limits(limits=1):
            model.fit(X_fit, y_fit, sample_weight=weights)
            pred = model.predict(X_valid_panel)
            proba = model.predict_proba(X_valid_panel)

        metrics = _post_classification_metrics(post_y_valid, pred, proba)
        original_summary = post_summary_by_rep[rep]
        post_performance_rows.append({
            "replicate_id": rep,
            "sampling_seed": int(original_summary["sampling_seed"]),
            "evaluation_design": design,
            "training_n": int(len(train_rows)),
            "selected_gene_count": int(len(genes)),
            "selected_cv_balanced_accuracy": float(original_summary["selected_cv_balanced_accuracy"]),
            **metrics,
        })

        for j, patient_id in enumerate(post_valid_ids):
            prediction_row = {
                "replicate_id": rep,
                "evaluation_design": design,
                "case_submitter_id": patient_id,
                "true_class": POST_LGG_CLASSES[int(post_y_valid[j])],
                "predicted_class": POST_LGG_CLASSES[int(pred[j])],
            }
            for class_index, class_name in enumerate(POST_LGG_CLASSES):
                prediction_row[f"probability_{class_name}"] = float(proba[j, class_index])
            post_prediction_rows.append(prediction_row)

post_performance = pd.DataFrame(post_performance_rows).sort_values(
    ["replicate_id", "evaluation_design"]
)
post_predictions = pd.DataFrame(post_prediction_rows)

post_metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "macro_ovr_roc_auc",
]
post_predictive_summary_rows = []
for design, group in post_performance.groupby("evaluation_design"):
    for metric in post_metric_columns:
        values = group[metric].dropna().to_numpy(dtype=float)
        post_predictive_summary_rows.append({
            "evaluation_design": design,
            "metric": metric,
            "n_replicates": int(len(values)),
            "mean": float(np.mean(values)),
            "sd": float(np.std(values, ddof=1)),
            "median": float(np.median(values)),
            "min": float(np.min(values)),
            "max": float(np.max(values)),
        })
post_predictive_summary = pd.DataFrame(post_predictive_summary_rows)

post_ba_wide = post_performance.pivot(
    index="replicate_id",
    columns="evaluation_design",
    values="balanced_accuracy",
).reset_index()
post_ba_wide["panel_only_minus_sampling_pipeline_balanced_accuracy"] = (
    post_ba_wide["panel_only_refit_197_patients"]
    - post_ba_wide["sampling_pipeline_157_patients"]
)
post_ba_wide = post_ba_wide.merge(
    post_performance.loc[
        post_performance["evaluation_design"] == "sampling_pipeline_157_patients",
        ["replicate_id", "selected_cv_balanced_accuracy", "selected_gene_count"],
    ],
    on="replicate_id",
    how="left",
)

post_performance.to_csv(
    POST_RESULTS_DIR / "LGG_20_resample_heldout_performance_by_replicate.csv", index=False
)
post_predictive_summary.to_csv(
    POST_RESULTS_DIR / "LGG_20_resample_heldout_performance_summary.csv", index=False
)
post_predictions.to_csv(
    POST_RESULTS_DIR / "LGG_20_resample_heldout_predictions_long.csv", index=False
)
post_ba_wide.to_csv(
    POST_RESULTS_DIR / "LGG_20_resample_balanced_accuracy_design_comparison.csv", index=False
)

# Publication-style predictive stability figure.
fig, ax = plt.subplots(figsize=(9.5, 5.8))
for design, label, marker in [
    ("sampling_pipeline_157_patients", "Fit on each 157-patient subsample", "o"),
    ("panel_only_refit_197_patients", "Panel refit on all 197 training patients", "s"),
]:
    g = post_performance.loc[post_performance["evaluation_design"] == design]
    ax.plot(
        g["replicate_id"],
        g["balanced_accuracy"],
        marker=marker,
        linewidth=1.5,
        markersize=5,
        label=label,
    )
ax.set_xlabel("Patient-resampling replicate")
ax.set_ylabel("Held-out balanced accuracy")
ax.set_xticks(post_replicate_ids)
ax.set_ylim(0.80, 1.01)
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(
    POST_RESULTS_DIR / "LGG_20_resample_heldout_balanced_accuracy.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    POST_RESULTS_DIR / "LGG_20_resample_heldout_balanced_accuracy.pdf",
    bbox_inches="tight",
)
plt.close(fig)

# ============================================================
# B. RUNNING STABILITY: FIRST 5, 10, 15, AND 20 RESAMPLES
# ============================================================

def _post_selection_matrix(panels, universe):
    lookup = {gene: j for j, gene in enumerate(universe)}
    labels = sorted(panels)
    Z = np.zeros((len(labels), len(universe)), dtype=np.uint8)
    for i, label in enumerate(labels):
        for gene in panels[label]:
            Z[i, lookup[gene]] = 1
    return Z


def _post_nogueira_stability(Z):
    Z = np.asarray(Z, dtype=float)
    M, d = Z.shape
    if M < 2:
        return np.nan
    p_hat = Z.mean(axis=0)
    sample_variance = (M / (M - 1.0)) * p_hat * (1.0 - p_hat)
    k_bar = Z.sum(axis=1).mean()
    denominator = (k_bar / d) * (1.0 - k_bar / d)
    if denominator <= 0:
        return np.nan
    return float(1.0 - sample_variance.mean() / denominator)


def _post_standardize_columns(X):
    X = np.asarray(X, dtype=np.float32)
    means = X.mean(axis=0, dtype=np.float64)
    centered = X - means.astype(np.float32)
    std = centered.std(axis=0, ddof=1, dtype=np.float64)
    if np.any(std <= 0):
        raise RuntimeError("At least one candidate gene has zero variance.")
    return centered / std.astype(np.float32)


post_corr_cache = POST_CACHE_DIR / "LGG_training_candidate_correlation_float32.npy"
if post_corr_cache.exists():
    print("Loading existing LGG correlation cache:", post_corr_cache)
    post_lgg_corr = np.load(post_corr_cache, mmap_mode="r")
else:
    print("LGG correlation cache not found. Recomputing once from the fixed 197-patient training cohort.")
    X_corr = post_train[post_candidate_genes].to_numpy(dtype=np.float32)
    Z_corr = _post_standardize_columns(X_corr)
    post_lgg_corr = ((Z_corr.T @ Z_corr) / np.float32(Z_corr.shape[0] - 1)).astype(np.float32)
    post_lgg_corr = np.clip(post_lgg_corr, -1.0, 1.0)
    np.save(post_corr_cache, post_lgg_corr)

post_abs_corr = np.abs(post_lgg_corr)
POST_PRIMARY_CORRELATION_THRESHOLD = 0.80


def _post_match_count(panel_a, panel_b, threshold=POST_PRIMARY_CORRELATION_THRESHOLD):
    idx_a = [post_gene_to_index[g] for g in panel_a]
    idx_b = [post_gene_to_index[g] for g in panel_b]
    M = np.asarray(post_abs_corr[np.ix_(idx_a, idx_b)], dtype=float)
    eligible = np.isfinite(M) & (M >= float(threshold))
    if not eligible.any():
        return 0
    score = np.where(eligible, 10_000.0 + M, 0.0)
    row_ind, col_ind = linear_sum_assignment(-score)
    return int(np.sum(eligible[row_ind, col_ind]))


post_running_rows = []
for n_runs in [5, 10, 15, 20]:
    ids = post_replicate_ids[:n_runs]
    current_panels = {rep: post_panels[rep] for rep in ids}
    Z = _post_selection_matrix(current_panels, post_candidate_genes)

    exact_jaccards = []
    corr_adjusted_jaccards = []
    matched_coverages = []

    for rep_a, rep_b in combinations(ids, 2):
        panel_a = post_panels[rep_a]
        panel_b = post_panels[rep_b]
        A, B = set(panel_a), set(panel_b)
        exact_jaccards.append(len(A & B) / len(A | B))

        matched_count = _post_match_count(panel_a, panel_b)
        corr_adjusted_jaccards.append(
            matched_count / (len(panel_a) + len(panel_b) - matched_count)
        )
        matched_coverages.append(matched_count / min(len(panel_a), len(panel_b)))

    post_running_rows.append({
        "resamples": int(n_runs),
        "pairwise_comparisons": int(len(exact_jaccards)),
        "mean_panel_size": float(np.mean([len(post_panels[r]) for r in ids])),
        "unique_selected_genes": int(len(set().union(*(set(post_panels[r]) for r in ids)))),
        "nogueira_stability": _post_nogueira_stability(Z),
        "mean_exact_jaccard": float(np.mean(exact_jaccards)),
        "mean_correlation_adjusted_jaccard_tau_0.80": float(np.mean(corr_adjusted_jaccards)),
        "mean_matched_coverage_tau_0.80": float(np.mean(matched_coverages)),
    })

post_running_stability = pd.DataFrame(post_running_rows)
post_running_stability.to_csv(
    POST_RESULTS_DIR / "LGG_running_stability_5_10_15_20.csv", index=False
)

fig, ax = plt.subplots(figsize=(8.5, 5.6))
ax.plot(
    post_running_stability["resamples"],
    post_running_stability["mean_exact_jaccard"],
    marker="o",
    linewidth=1.7,
    label="Mean exact Jaccard",
)
ax.plot(
    post_running_stability["resamples"],
    post_running_stability["nogueira_stability"],
    marker="s",
    linewidth=1.7,
    label="Nogueira stability",
)
ax.plot(
    post_running_stability["resamples"],
    post_running_stability["mean_correlation_adjusted_jaccard_tau_0.80"],
    marker="^",
    linewidth=1.7,
    label="Mean correlation-adjusted Jaccard (|r| ≥ 0.80)",
)
ax.set_xlabel("Number of patient-resampling runs included")
ax.set_ylabel("Stability statistic")
ax.set_xticks([5, 10, 15, 20])
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(
    POST_RESULTS_DIR / "LGG_running_stability_5_10_15_20.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    POST_RESULTS_DIR / "LGG_running_stability_5_10_15_20.pdf",
    bbox_inches="tight",
)
plt.close(fig)

# ============================================================
# C. FORMAL GENE RECURRENCE ACROSS ALL 20 RESAMPLES
# ============================================================
post_recurrence_counts = Counter()
post_gene_replicates = {}
for rep, genes in post_panels.items():
    for gene in genes:
        post_recurrence_counts[gene] += 1
        post_gene_replicates.setdefault(gene, []).append(rep)

post_gene_recurrence = pd.DataFrame([
    {
        "gene_id": gene,
        "gene_name": post_gene_to_symbol.get(gene, ""),
        "selection_count": int(count),
        "selection_fraction": float(count / len(post_panels)),
        "replicate_ids": ";".join(map(str, sorted(post_gene_replicates[gene]))),
    }
    for gene, count in post_recurrence_counts.items()
]).sort_values(
    ["selection_count", "gene_name", "gene_id"],
    ascending=[False, True, True],
).reset_index(drop=True)

post_recurrence_distribution = (
    post_gene_recurrence.groupby("selection_count")
    .size()
    .reset_index(name="number_of_genes")
    .sort_values("selection_count")
)
post_recurrence_distribution["fraction_of_distinct_selected_genes"] = (
    post_recurrence_distribution["number_of_genes"] / len(post_gene_recurrence)
)

post_total_selection_slots = int(sum(len(panel) for panel in post_panels.values()))
post_recurrence_summary = {
    "completed_resamples": int(len(post_panels)),
    "total_selection_slots": post_total_selection_slots,
    "distinct_selected_genes": int(len(post_gene_recurrence)),
    "genes_selected_once": int((post_gene_recurrence["selection_count"] == 1).sum()),
    "genes_selected_twice": int((post_gene_recurrence["selection_count"] == 2).sum()),
    "genes_selected_three_or_more_times": int((post_gene_recurrence["selection_count"] >= 3).sum()),
    "maximum_recurrence_count": int(post_gene_recurrence["selection_count"].max()),
    "maximum_recurrence_fraction": float(post_gene_recurrence["selection_fraction"].max()),
}

post_gene_recurrence.to_csv(
    POST_RESULTS_DIR / "LGG_20_resample_gene_recurrence.csv", index=False
)
post_recurrence_distribution.to_csv(
    POST_RESULTS_DIR / "LGG_20_resample_gene_recurrence_distribution.csv", index=False
)
with open(
    POST_RESULTS_DIR / "LGG_20_resample_gene_recurrence_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(post_recurrence_summary, handle, indent=2)

# Top recurrent genes figure. A top-20 display is descriptive only; recurrence
# is not treated as proof of biological importance.
post_top_recurrence = post_gene_recurrence.head(20).copy()
post_top_recurrence = post_top_recurrence.iloc[::-1]
fig, ax = plt.subplots(figsize=(8.5, 7.0))
ax.barh(post_top_recurrence["gene_name"], post_top_recurrence["selection_count"])
ax.set_xlabel("Number of the 20 resampling panels selecting the gene")
ax.set_ylabel("Gene")
ax.set_xlim(0, 20)
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(
    POST_RESULTS_DIR / "LGG_top20_gene_recurrence_across_resamples.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    POST_RESULTS_DIR / "LGG_top20_gene_recurrence_across_resamples.pdf",
    bbox_inches="tight",
)
plt.close(fig)

# ============================================================
# D. COMPACT MANUSCRIPT-READY SUMMARY
# ============================================================
post_pipeline_ba = post_performance.loc[
    post_performance["evaluation_design"] == "sampling_pipeline_157_patients",
    "balanced_accuracy",
]
post_panelonly_ba = post_performance.loc[
    post_performance["evaluation_design"] == "panel_only_refit_197_patients",
    "balanced_accuracy",
]
post_run20 = post_running_stability.loc[post_running_stability["resamples"] == 20].iloc[0]

post_key_summary = pd.DataFrame([{
    "experiment": "TCGA-LGG 20 patient-resampling panels",
    "completed_resamples": 20,
    "total_selection_slots": post_total_selection_slots,
    "distinct_selected_genes": len(post_gene_recurrence),
    "nogueira_stability": post_run20["nogueira_stability"],
    "mean_exact_jaccard": post_run20["mean_exact_jaccard"],
    "mean_corr_adjusted_jaccard_tau_0.80": post_run20["mean_correlation_adjusted_jaccard_tau_0.80"],
    "mean_matched_coverage_tau_0.80": post_run20["mean_matched_coverage_tau_0.80"],
    "sampling_pipeline_mean_heldout_balanced_accuracy": post_pipeline_ba.mean(),
    "sampling_pipeline_sd_heldout_balanced_accuracy": post_pipeline_ba.std(ddof=1),
    "sampling_pipeline_min_heldout_balanced_accuracy": post_pipeline_ba.min(),
    "sampling_pipeline_max_heldout_balanced_accuracy": post_pipeline_ba.max(),
    "panel_only_mean_heldout_balanced_accuracy": post_panelonly_ba.mean(),
    "panel_only_sd_heldout_balanced_accuracy": post_panelonly_ba.std(ddof=1),
    "panel_only_min_heldout_balanced_accuracy": post_panelonly_ba.min(),
    "panel_only_max_heldout_balanced_accuracy": post_panelonly_ba.max(),
}])
post_key_summary.to_csv(
    POST_TABLE_DIR / "Paper2_LGG_20_resample_final_summary.csv", index=False
)

# Update the existing overall key-results table if it is already present.
existing_key_path = POST_TABLE_DIR / "Paper2_key_results_summary.csv"
if existing_key_path.exists():
    existing_key = pd.read_csv(existing_key_path)
    existing_key = existing_key.loc[
        existing_key["dataset_experiment"] != "TCGA-LGG 20 patient resamples"
    ].copy()
    sampling_row = {
        "dataset_experiment": "TCGA-LGG 20 patient resamples",
        "perturbation": "patient sampling",
        "common_candidate_universe": True,
        "candidate_universe_size": 3000,
        "number_of_panels": 20,
        "panel_size": float(np.mean([len(p) for p in post_panels.values()])),
        "unique_selected_genes": len(post_gene_recurrence),
        "mean_exact_jaccard": post_run20["mean_exact_jaccard"],
        "nogueira_stability": post_run20["nogueira_stability"],
        "mean_corr_adjusted_jaccard_tau_0.80": post_run20["mean_correlation_adjusted_jaccard_tau_0.80"],
        "mean_chance_corrected_corr_adjusted_stability_tau_0.80": np.nan,
        "global_null_largest_component_size": np.nan,
        "global_null_large_components_seeds_ge4": np.nan,
        # For comparability with the five-seed row, this field uses the panel-only
        # refit on the same full 197-patient training cohort.
        "mean_fixed_HGB_heldout_balanced_accuracy": float(post_panelonly_ba.mean()),
    }
    existing_key = pd.concat([existing_key, pd.DataFrame([sampling_row])], ignore_index=True)
    existing_key.to_csv(existing_key_path, index=False)

# Record completion without disturbing the original analysis configuration fields.
config_path = POST_OUTPUT_DIR / "Paper2_analysis_configuration.json"
if config_path.exists():
    try:
        config = json.loads(config_path.read_text(encoding="utf-8"))
    except Exception:
        config = {}
else:
    config = {}
config["final_20_resample_post_analysis_completed"] = True
config["final_20_resample_post_analysis_designs"] = [
    "sampling_pipeline_157_patients_to_85_heldout",
    "panel_only_refit_197_patients_to_85_heldout",
    "running_stability_first_5_10_15_20_resamples",
    "gene_recurrence_across_20_resamples",
]
config_path.write_text(json.dumps(config, indent=2, default=str), encoding="utf-8")

print("\nFinal 20-resample post-analysis complete.")
print("Outputs saved to:", POST_RESULTS_DIR)
print("\nHeld-out balanced accuracy summary:")
display(
    post_predictive_summary.loc[
        post_predictive_summary["metric"] == "balanced_accuracy"
    ].reset_index(drop=True)
)
print("\nRunning stability summary:")
display(post_running_stability)
print("\nTop recurrent genes:")
display(post_gene_recurrence.head(15))
print("\nCompact final summary:")
display(post_key_summary)


## 27. Consolidate all figures into `PAPER2_OUTPUTS/FIGURES`

This section keeps every figure in its original analysis-specific folder **and** copies a second copy into the top-level `FIGURES` folder for manuscript preparation. It does not rerun any analysis. A manifest is also written so each consolidated figure can be traced back to its original source folder.


In [ ]:
# ============================================================
# CONSOLIDATE ALL GENERATED FIGURES FOR MANUSCRIPT PREPARATION
# ============================================================
# This cell does NOT rerun any analysis.
# It scans PAPER2_OUTPUTS recursively for figure files, leaves the
# originals in place, and copies them into PAPER2_OUTPUTS/FIGURES.
#
# Run this cell any time new figures have been generated.

from pathlib import Path
import shutil
import os
import pandas as pd

# Resolve PAPER2_OUTPUTS robustly whether or not earlier notebook cells
# were run in the current session.
if "OUTPUT_DIR" in globals():
    CONSOLIDATE_OUTPUT_DIR = Path(OUTPUT_DIR)
elif "POST_OUTPUT_DIR" in globals():
    CONSOLIDATE_OUTPUT_DIR = Path(POST_OUTPUT_DIR)
else:
    _default_base = Path(
        r"C:\Users\we3mo\Dropbox\MY DATA DRIVE\CANCER"
        r"\2026 - ADD ANALYSIS FOR TCGA BIOINFORMATICS_2"
    )
    _base = Path(os.environ.get("PAPER2_BASE_DIR", str(_default_base)))
    CONSOLIDATE_OUTPUT_DIR = _base / "PAPER2_OUTPUTS"

if not CONSOLIDATE_OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"PAPER2_OUTPUTS was not found at:\n{CONSOLIDATE_OUTPUT_DIR}"
    )

CONSOLIDATED_FIGURES_DIR = CONSOLIDATE_OUTPUT_DIR / "FIGURES"
CONSOLIDATED_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_EXTENSIONS = {".png", ".pdf", ".svg", ".tif", ".tiff", ".eps"}

# Remove only previously consolidated figure copies and the prior manifest.
# Original figures in the analysis-specific folders are never touched.
for existing in CONSOLIDATED_FIGURES_DIR.iterdir():
    if existing.is_file() and (
        existing.suffix.lower() in FIGURE_EXTENSIONS
        or existing.name == "Paper2_consolidated_figure_manifest.csv"
    ):
        existing.unlink()

source_figures = []
for p in CONSOLIDATE_OUTPUT_DIR.rglob("*"):
    if not p.is_file():
        continue
    if p.suffix.lower() not in FIGURE_EXTENSIONS:
        continue
    # Do not recursively copy files that are already in the consolidated folder.
    if CONSOLIDATED_FIGURES_DIR in p.parents:
        continue
    source_figures.append(p)

source_figures = sorted(source_figures)

manifest_rows = []
used_names = set()

for src_path in source_figures:
    rel = src_path.relative_to(CONSOLIDATE_OUTPUT_DIR)

    # Prefer the original filename when it is unique.
    dest_name = src_path.name

    # If another analysis folder contains a figure with the same filename,
    # prefix the source folder path so neither copy is overwritten.
    if dest_name.lower() in used_names:
        parent_tag = "__".join(rel.parts[:-1])
        parent_tag = parent_tag.replace(" ", "_")
        dest_name = f"{parent_tag}__{src_path.name}"

        # Final safeguard against any remaining collision.
        counter = 2
        stem = Path(dest_name).stem
        suffix = Path(dest_name).suffix
        while dest_name.lower() in used_names:
            dest_name = f"{stem}__{counter}{suffix}"
            counter += 1

    used_names.add(dest_name.lower())

    dest_path = CONSOLIDATED_FIGURES_DIR / dest_name
    shutil.copy2(src_path, dest_path)

    manifest_rows.append({
        "consolidated_filename": dest_name,
        "source_relative_path": str(rel),
        "file_type": src_path.suffix.lower(),
        "size_bytes": dest_path.stat().st_size,
    })

figure_manifest = pd.DataFrame(manifest_rows)
manifest_path = CONSOLIDATED_FIGURES_DIR / "Paper2_consolidated_figure_manifest.csv"
figure_manifest.to_csv(manifest_path, index=False)

print(f"Found {len(source_figures):,} figure files outside the consolidated FIGURES folder.")
print(f"Copied {len(figure_manifest):,} figure files to:")
print(CONSOLIDATED_FIGURES_DIR)
print(f"\nManifest saved to:\n{manifest_path}")

if len(figure_manifest):
    display(figure_manifest)
else:
    print(
        "\nNo figure files were found. If this is unexpected, first run the "
        "figure-generating analysis cells and then rerun this consolidation cell."
    )


## 28. Output inventory


In [ ]:
output_files = sorted([p for p in OUTPUT_DIR.rglob("*") if p.is_file()])
inventory = pd.DataFrame({
    "relative_path": [str(p.relative_to(OUTPUT_DIR)) for p in output_files],
    "size_bytes": [p.stat().st_size for p in output_files],
})
inventory.to_csv(OUTPUT_DIR / "Paper2_output_file_inventory.csv", index=False)
print(f"Files currently present beneath PAPER2_OUTPUTS: {len(inventory):,}")
display(inventory.head(100))